# 0. Global loads

In [78]:
# imports
import pickle, pandas as pd, numpy as np
from matplotlib import pyplot as plt
from matplotlib.transforms import blended_transform_factory
import seaborn as sns
from tqdm import tqdm
import re
import copy
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, accuracy_score, f1_score, make_scorer
from sklearn.model_selection import ParameterGrid
from scipy import stats
from scipy.stats import t
from scipy.interpolate import make_interp_spline
from typing import List
from collections import defaultdict
from metapaths.starterpack import load_pickle, save_pickle, cypher_triple_to_list, metapath_featset_gen
from metapaths.inf import INFToolbox, query_templates_234, Graph
from typing import Dict, Any, List, Tuple, Optional
import config

In [ ]:
# reference copied from feat extraction notebook

rel_types_cypher = ["(:Gene)-[:Gene_Gene]->(:Gene)",
"(:Gene)-[:Gene_Disease]->(:Disease)",
"(:Gene)-[:Gene_MolecularFunction]->(:MolecularFunction)",
"(:Gene)-[:Gene_BiologicalProcess]->(:BiologicalProcess)",
"(:Gene)-[:Gene_CellularComponent]->(:CellularComponent)",
"(:Gene)-[:Gene_Pathway]->(:Pathway)",
"(:Gene)-[:Gene_Anatomy]->(:Anatomy)",
"(:Gene)-[:Gene_Compound]->(:Compound)",
"(:Disease)-[:Disease_Disease]->(:Disease)",
"(:Compound)-[:Compound_Disease]->(:Disease)",
"(:Compound)-[:Compound_SideEffect]->(:SideEffect)",
"(:CellularComponent)-[:CellularComponent_CellularComponent]->(:CellularComponent)",
"(:BiologicalProcess)-[:BiologicalProcess_BiologicalProcess]->(:BiologicalProcess)",
"(:MolecularFunction)-[:MolecularFunction_MolecularFunction]->(:MolecularFunction)",
"(:Disease)-[:Disease_Symptom]->(:Symptom)",
"(:Compound)-[:Compound_Compound]->(:Compound)",
"(:ADR)-[:ADR_Disease]->(:Disease)",
"(:ADR)-[:ADR_Symptom]->(:Symptom)",
"(:Gene)-[:Gene_ADR]->(:ADR)",
"(:ADR)-[:ADR_SideEffect]->(:SideEffect)"]

len(rel_types_cypher)

In [81]:
pickles_dir = str(config.PICKLES_DIR)

In [82]:
# load correct db
try:
    toolbox = INFToolbox(Graph(config.NEO4J_URI_READ), query_templates_234)
    _toolbox_ok = True
except Exception as _neo4j_err:
    print(f"Neo4j unavailable ({type(_neo4j_err).__name__}): {_neo4j_err}")
    print("All pre-computed pickles will be used; toolbox set to None.")
    toolbox = None
    _toolbox_ok = False

param_combos_load = load_pickle(f'{pickles_dir}/param_combos.pkl')
if _toolbox_ok:
    toolbox.add_param_combos(param_combos_load)

In [95]:
all_genes_aes = load_pickle(f'{pickles_dir}/all_genes_with_aes.pkl')

# I. Reproduce CV with Grid Search

### CV function

In [ ]:
def cv_gs_loop(folds_feats: dict, parameter_name: str, cols_to_drop: list, scoring_metric: str = 'recall'):

    # Define the hyperparameter grid to search over.
    param_grid = {
        'max_features': ['sqrt', 'log2'],
        'max_depth': [20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    # Helper to map the scoring metric string to the correct key in the results dict.
    metric_key_map = {
        'recall': 'overall_recall',
        'precision': 'overall_precision',
        'f1': 'overall_f1'
    }
    if scoring_metric not in metric_key_map:
        raise ValueError("scoring_metric must be one of 'recall', 'precision', or 'f1'")

    # Initialize variables to track the best model found so far.
    best_score = -1.0
    best_params = None
    best_fold_results = None

    # Use ParameterGrid to create all combinations of hyperparameters.
    grid = ParameterGrid(param_grid)

    # --- Grid Search Loop ---
    # Iterate over each unique combination of hyperparameters.
    for params in grid:
        current_fold_results = {}
        scores_for_current_params = []

        # --- Cross-Validation Loop ---
        # For the current hyperparameters, iterate over each fold of the data.
        for fold_id, fold_data in folds_feats.items():
            # Prepare training data by combining positive and negative samples.
            X_train_pos = fold_data['pos_train'].drop([parameter_name] + cols_to_drop, axis=1)
            X_train_neg = fold_data['neg_train'].drop([parameter_name] + cols_to_drop, axis=1)
            X_train = np.vstack([X_train_pos.values, X_train_neg.values])
            y_train = np.hstack([np.ones(len(X_train_pos)), np.zeros(len(X_train_neg))])

            # Prepare validation data similarly.
            X_val_pos = fold_data['pos_val'].drop([parameter_name] + cols_to_drop, axis=1)
            X_val_neg = fold_data['neg_val'].drop([parameter_name] + cols_to_drop, axis=1)
            X_val = np.vstack([X_val_pos.values, X_val_neg.values])
            y_val = np.hstack([np.ones(len(X_val_pos)), np.zeros(len(X_val_neg))])

            # Extract parameter values from the validation set for later analysis.
            all_param_values = np.concatenate([
                fold_data['pos_val'][parameter_name].values,
                fold_data['neg_val'][parameter_name].values
            ])

            # Instantiate the model with the current set of hyperparameters.
            model = RandomForestClassifier(n_estimators=200, random_state=69, **params)
            
            # Train the model and make predictions on the validation set.
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            
            # Store all performance metrics for the current fold.
            current_fold_results[fold_id] = {
                'y_val': y_val, 'y_pred': y_pred,
                'parameter_values': all_param_values,
                'n_pos_val': len(X_val_pos),
                'overall_recall': recall_score(y_val, y_pred, zero_division=0),
                'overall_precision': precision_score(y_val, y_pred, zero_division=0),
                'overall_f1': f1_score(y_val, y_pred, zero_division=0)
            }
            # Append the primary score for this fold to track performance of this hyperparameter set.
            scores_for_current_params.append(current_fold_results[fold_id][metric_key_map[scoring_metric]])

        # After all folds, calculate the mean score for the current hyperparameters.
        mean_score = np.mean(scores_for_current_params)

        # --- Best Model Selection ---
        # If the current model's average score is the best so far, update trackers.
        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            best_fold_results = current_fold_results

    # --- Final Metrics Calculation ---
    # After the grid search, calculate aggregate metrics for the single best model.
    if best_fold_results is None:
        raise RuntimeError("Grid search completed without finding a best model.")

    recalls = [r['overall_recall'] for r in best_fold_results.values()]
    precisions = [r['overall_precision'] for r in best_fold_results.values()]
    f1_scores = [r['overall_f1'] for r in best_fold_results.values()]
    
    # Package the final results for the best model.
    final_metrics = (
        np.mean(recalls), np.std(recalls),
        np.mean(precisions), np.std(precisions),
        np.mean(f1_scores), np.std(f1_scores)
    )

    return final_metrics, best_fold_results, best_params

### Quantiles as per pd.qcut

In [30]:
def ci_margin_of_error(data: pd.Series, confidence: float = 0.95) -> float:
    """Helper function to calculate the confidence interval margin of error."""
    # CI requires at least 2 data points
    if len(data) < 2:
        return 0.0
    
    # Calculate sample standard deviation (ddof=1) and count
    std_dev = np.std(data, ddof=1)
    n = len(data)
    
    # Get the t-score
    t_score = t.ppf((1 + confidence) / 2, df=n - 1)
    
    # Calculate the margin of error
    margin = t_score * (std_dev / np.sqrt(n))
    return margin

In [44]:
def analyse_parameter_slices_quantile(fold_results: dict, num_slices: int = 10, balance_slices: bool = False, parameter_name: str = 'Gene_ADR_deg', confidence_level: float = 0.95):
    """
    Amended: Analyzes performance by slicing positives into quantiles.
    """
    slice_data = []
    RANDOM_SEED = 69
    for fold_id, fold_data in fold_results.items():
        # --- AMENDED: Use the parameter_name argument to find the right data ---
        fold_df = pd.DataFrame({
            'y_val': fold_data['y_val'],
            'y_pred': fold_data['y_pred'],
            'param_values': fold_data.get(parameter_name, fold_data.get('parameter_values'))
        })
        pos_df = fold_df[fold_df['y_val'] == 1].copy()
        if pos_df.empty: continue

        try:
            pos_df['slice_num'] = pd.qcut(pos_df['param_values'], q=num_slices, labels=False, duplicates='drop')
        except ValueError: continue

        for slice_id in sorted(pos_df['slice_num'].unique()):
            slice_pos_df = pos_df[pos_df['slice_num'] == slice_id]
            min_param, max_param = slice_pos_df['param_values'].min(), slice_pos_df['param_values'].max()
            n_pos = len(slice_pos_df)
            slice_neg_conformant = fold_df[(fold_df['y_val'] == 0) & (fold_df['param_values'].between(min_param, max_param))]
            final_slice_negs = slice_neg_conformant.sample(n=n_pos, random_state=RANDOM_SEED) if balance_slices and len(slice_neg_conformant) > n_pos else slice_neg_conformant
            final_slice_df = pd.concat([slice_pos_df, final_slice_negs])
            y_true, y_pred = final_slice_df['y_val'], final_slice_df['y_pred']
            
            slice_data.append({
                'slice_num': slice_id + 1, 'recall': recall_score(y_true, y_pred, zero_division=0),
                'precision': precision_score(y_true, y_pred, zero_division=0), 'f_1': f1_score(y_true, y_pred, zero_division=0),
                'min_param': min_param, 'max_param': max_param, 'true_pos': np.sum((y_pred == 1) & (y_true == 1)),
                'pred_pos': np.sum(y_pred), 'val_neg': len(final_slice_negs), 'pos': n_pos
            })

    if not slice_data: return pd.DataFrame()
    df = pd.DataFrame(slice_data)
    
    # --- AMENDED: Inserted data cleaning block to fix TypeError in .agg() ---
    cols_to_convert = ['recall', 'precision', 'f_1', 'true_pos', 'pred_pos', 'val_neg', 'pos', 'min_param', 'max_param']
    for col in cols_to_convert:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=[c for c in cols_to_convert if c in df.columns], inplace=True)
    if df.empty: return pd.DataFrame()

    ci_func = lambda x: ci_margin_of_error(x, confidence=confidence_level)
    result = df.groupby('slice_num').agg(
        recall_mean=('recall', 'mean'), recall_ci=('recall', ci_func),
        precision_mean=('precision', 'mean'), precision_ci=('precision', ci_func),
        f1_mean=('f_1', 'mean'), f1_ci=('f_1', ci_func),
        mean_true_pos=('true_pos', 'mean'), mean_pred_pos=('pred_pos', 'mean'),
        mean_val_neg=('val_neg', 'mean'), mean_pos_per_slice=('pos', 'mean'),
        min_param=('min_param', 'mean'), max_param=('max_param', 'mean')
    ).reset_index()

    result[['min_param', 'max_param']] = result[['min_param', 'max_param']].round(0).astype(int)
    return result.round(3)

In [516]:
def visualise_quantile_boundaries(
    data_series: pd.Series,
    num_slices: int = 10,
    title: str = "Distribution with Quantile Boundaries"
):
    """
    Calculates and visualizes quantile boundaries over the data's distribution
    using a histogram for improved clarity and reduced visual clutter.

    Args:
        data_series (pd.Series): The numerical data to be sliced.
        num_slices (int): The number of quantiles to create (e.g., 10 for deciles).
        title (str): The title for the plot.
    """
    # --- 1. Calculate Quantile Boundaries ---
    try:
        _, boundaries = pd.qcut(data_series, q=num_slices, labels=False, retbins=True, duplicates='drop')
    except ValueError:
        print("Could not generate bins. The data may have too few unique values.")
        return

    num_generated_slices = len(boundaries) - 1
    print(f"Number of slices requested: {num_slices}, Generated: {num_generated_slices}")
    print(f"Boundary values: {np.round(boundaries, 2)}")

    # --- 2. Create the Visualization (Histogram Only) ---
    sns.set_theme(style="ticks") # A clean style without a background grid
    plt.figure(figsize=(24, 7))

    # Plot the histogram without the KDE curve
    ax = sns.histplot(data_series, kde=False, color="#5F9EA0", bins=50)

    # --- 3. Format Axes for Clarity ---
    ax.set_xlim(left=0)  # Ensure x-axis starts at 0
    ax.xaxis.grid(False) # Remove vertical grid lines
    ax.yaxis.grid(True, linestyle='--', alpha=0.6) # Keep horizontal grid lines
    ax.tick_params(axis='both', which='major', labelsize=12) # Increase tick font size

    # Add quantile boundaries as ticks on the x-axis
    ax.set_xticks(boundaries)
    plt.xticks(rotation=30, ha='right') # Rotate labels to prevent overlap

    # --- 4. Add Boundary Lines and Staggered Annotations ---
    y_max = ax.get_ylim()[1]
    # Define two alternating heights for the text labels
    y_pos_high = y_max * 0.9
    y_pos_low = y_max * 0.8

    # Iterate through interior boundaries
    for i, boundary in enumerate(boundaries[1:-1], 1):
        ax.axvline(x=boundary, color="#B22222", linestyle="--", linewidth=1.5)

        # Alternate the vertical position of the text to prevent overlap
        y_pos = y_pos_high if i % 2 != 0 else y_pos_low
        percentile = (i / num_generated_slices) * 100
        ax.text(
            boundary,
            y_pos,
            f"{percentile:.0f}%",
            color="#333333",
            ha='center',
            fontweight='bold'
        )

    # --- 5. Final Touches ---
    ax.set_title(title, fontsize=18, fontweight='bold', pad=20)
    ax.set_xlabel(data_series.name, fontsize=14)
    ax.set_ylabel("Frequency", fontsize=14)
    plt.tight_layout()
    plt.show()

In [32]:
def find_best_configs_quantile(config_results_dict, parameter_name, rank,
    num_slices: int = 10, balance_slices: bool = False, metric: str = 'recall_mean'):
    """
    Amended: Finds the Nth best-performing config for each parameter slice.
    """
    if not isinstance(rank, int) or rank < 1:
        raise ValueError("Rank must be a positive integer (1 or greater).")

    all_configs_data = []
    for config_idx, (_, fold_results, _) in config_results_dict.items():
        # --- AMENDED: Pass the 'parameter_name' argument to the analysis function ---
        slice_df = analyse_parameter_slices_quantile(fold_results, num_slices, balance_slices, parameter_name)
        if slice_df.empty: continue
        
        slice_df.insert(1, 'param_combo', config_idx)
        all_configs_data.append(slice_df)
        
    if not all_configs_data: return pd.DataFrame()
    combined_df = pd.concat(all_configs_data, ignore_index=True)
    if combined_df.empty or combined_df[metric].isna().all(): return pd.DataFrame()
        
    return (
        combined_df.sort_values(by=metric, ascending=False)
        .groupby('slice_num').nth(rank - 1).reset_index()
    )

## 1. L-3

In [10]:
folds_l3_inf_feats_combos = load_pickle(str(config.CV_INF_FEATS_L3_PKL))

In [ ]:
folds_l3_inf_feats_combos[0]['fold_0']['neg_train']['Gene_ADR_deg'].value_counts()[:10]

In [ ]:
folds_l3_inf_feats_combos[0]['fold_0']['neg_train']['Gene_ADR_deg'].value_counts()[:10].sum()/5327

In [ ]:
visualise_quantile_boundaries(folds_l3_inf_feats_combos[0]['fold_0']['pos_val']['Gene_ADR_deg'])

### CV L3

In [ ]:
from pathlib import Path as _Path
if _Path(config.CV_GS_RESULTS_L3_PKL).exists():
    inf_cv_gs_results_l3 = load_pickle(str(config.CV_GS_RESULTS_L3_PKL))
    print(f'Pre-computed — loaded {config.CV_GS_RESULTS_L3_PKL.name}')
else:
    # Recall
    inf_cv_gs_results_l3 = {}

    for combo_idx, combo in tqdm(enumerate(toolbox.param_combos), total=len(toolbox.param_combos)):

        inf_cv_gs_results_l3[combo_idx] = cv_gs_loop(folds_l3_inf_feats_combos[combo_idx], 'Gene_ADR_deg', ['Gene','ADR'])


    save_pickle(inf_cv_gs_results_l3, str(config.CV_GS_RESULTS_L3_PKL))

In [ ]:
from pathlib import Path as _Path
if _Path(config.CV_GS_RESULTS_L3_BOTTOM_PKL).exists():
    inf_cv_gs_results_l3_bottom = load_pickle(str(config.CV_GS_RESULTS_L3_BOTTOM_PKL))
    print(f'Pre-computed — loaded {config.CV_GS_RESULTS_L3_BOTTOM_PKL.name}')
else:
    # Recall bottom 2 slices -- report after first-pass CV (ie no need to do for L4c)
    inf_cv_gs_results_l3_bottom = {}

    for combo_idx, combo in tqdm(enumerate(toolbox.param_combos), total=len(toolbox.param_combos)):

        inf_cv_gs_results_l3_bottom[combo_idx] = cv_gs_loop_for_edge_distribution_bottom(folds_l3_inf_feats_combos[combo_idx], 'Gene_ADR_deg', ['Gene','ADR'])


    save_pickle(inf_cv_gs_results_l3_bottom, str(config.CV_GS_RESULTS_L3_BOTTOM_PKL))

#### Raw config results

In [ ]:
# Raw counts slice results
analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1])[['min_param','max_param',
                                                               'recall_mean', 'recall_ci',
                                                               'mean_pos_per_slice', 
                                                               'mean_true_pos', 'mean_pred_pos',
                                                               'precision_mean', 'precision_ci',
                                                               'f1_mean', 'f1_ci',
                                                               'mean_val_neg']]

### L3 best configs; ranks 1-5

In [ ]:
find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 1)[['min_param','max_param',
                                                                  'param_combo','recall_mean', 'recall_ci',
                                                                  'mean_pos_per_slice', 
                                                                  'mean_true_pos', 'mean_pred_pos',
                                                                  'precision_mean', 'precision_ci',
                                                                  'f1_mean', 'f1_ci',
                                                                  'mean_val_neg']]

In [ ]:
# 2nd ranked configs (recall)
find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 2)[['param_combo','recall_mean', 'recall_ci',
                                                                  'mean_pos_per_slice', 
                                                                  'mean_true_pos', 'mean_pred_pos',
                                                                  'precision_mean', 'precision_ci',
                                                                  'f1_mean', 'f1_ci',
                                                                  'mean_val_neg']]

In [ ]:
# 3rd ranked configs (recall)
find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 3)[['param_combo','recall_mean', 'recall_ci',
                                                                  'mean_pos_per_slice', 
                                                                  'mean_true_pos', 'mean_pred_pos',
                                                                  'precision_mean', 'precision_ci',
                                                                  'f1_mean', 'f1_ci',
                                                                  'mean_val_neg']]

In [ ]:
# 4th ranked configs (recall)
find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 4)[['param_combo','recall_mean', 'recall_ci',
                                                                  'mean_pos_per_slice', 
                                                                  'mean_true_pos', 'mean_pred_pos',
                                                                  'precision_mean', 'precision_ci',
                                                                  'f1_mean', 'f1_ci',
                                                                  'mean_val_neg']]

In [ ]:
for i in range(1,6):

    print(f'Rank {i} configs for slice 1:')
    print(param_combos_load[find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', i)['param_combo'][0]])
    print('\n')

print('\n')

for i in range(1,6):

    print(f'Rank {i} configs for slice 2:')
    print(param_combos_load[find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', i)['param_combo'][1]])

In [ ]:
# 5th ranked configs (recall)
find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 5)[['param_combo','recall_mean', 'recall_ci',
                                                                  'mean_pos_per_slice', 
                                                                  'mean_true_pos', 'mean_pred_pos',
                                                                  'precision_mean', 'precision_ci',
                                                                  'f1_mean', 'f1_ci',
                                                                  'mean_val_neg']]

### INF vs Raw Comparison Plot

In [222]:
def plot_smoothed_recall(dfs: List[pd.DataFrame], labels: List[str], title: str, xlabel: str = "Data Decile", ylabel: str = "Mean Recall" # CIs capped at 1.0
):
    """
    Generates smoothed line plots of recall and its confidence interval.

    Args:
        dfs (List[pd.DataFrame]): A list of DataFrames, each with the required format
                                  containing 'recall_mean', 'recall_ci', 'min_param',
                                  and 'max_param' columns.
        labels (List[str]): A list of labels for each DataFrame, used in the legend.
        title (str): The title of the chart.
        xlabel (str, optional): The label for the x-axis.
        ylabel (str, optional): The label for the y-axis.
    """
    # --- 1. Setup Chart and Colors ---
    fig, ax = plt.subplots(figsize=(12, 8))
    
    cmap = plt.get_cmap('tab10')
    colors = cmap.colors

    # --- 2. Iterate and Plot Each DataFrame ---
    for i, df in enumerate(dfs):
        df = df.sort_index()
        
        x_orig = df.index
        y_recall = df['recall_mean']
        y_ci = df['recall_ci']

        x_smooth = np.linspace(x_orig.min(), x_orig.max(), 300)

        # --- 3. Create Smoothed Lines using Spline Interpolation ---
        spline_recall = make_interp_spline(x_orig, y_recall, k=3)
        y_smooth_recall = spline_recall(x_smooth)

        spline_upper = make_interp_spline(x_orig, y_recall + y_ci, k=3)
        y_smooth_upper = spline_upper(x_smooth)
        
        spline_lower = make_interp_spline(x_orig, y_recall - y_ci, k=3)
        y_smooth_lower = spline_lower(x_smooth)

        # --- AMENDED SECTION: Cap values at 0 and 1 ---
        # Ensure that smoothed lines and CIs do not exceed the logical bounds of [0, 1]
        y_smooth_recall = np.clip(y_smooth_recall, 0, 1.0)
        y_smooth_upper = np.clip(y_smooth_upper, 0, 1.0)
        y_smooth_lower = np.clip(y_smooth_lower, 0, 1.0)
        
        # --- 4. Plot the Data ---
        ax.plot(x_smooth, y_smooth_recall, label=labels[i], color=colors[i % len(colors)], linewidth=2.5)
        ax.fill_between(x_smooth, y_smooth_lower, y_smooth_upper, color=colors[i % len(colors)], alpha=0.15)

    # --- 5. Format X-axis Ticks ---
    ref_df = dfs[0]
    tick_positions = ref_df.index
    tick_labels = [
        f"{i + 1}\n({int(row.min_param)}-{int(row.max_param)})"
        for i, row in ref_df.iterrows()
    ]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=11)

    # --- 6. Final Chart Styling ---
    ax.set_title(title, fontsize=20, pad=20)
    ax.set_xlabel(xlabel, fontsize=16, labelpad=15)
    ax.set_ylabel(ylabel, fontsize=16, labelpad=15)
    ax.legend(fontsize=14, frameon=True, shadow=True)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    # Set a firm y-axis limit from 0 to 1, with a small margin for aesthetics
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='both', which='major', labelsize=12)
    
    plt.tight_layout()
    plt.show()

In [95]:
best_configs_l3_recall = find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 1)[['min_param', 'max_param', 'recall_mean', 'recall_ci']]
raw_l3_recall = analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1])[['min_param', 'max_param', 'recall_mean', 'recall_ci']]

In [1567]:
if not (config.BEST_CONFIGS_L3_RECALL_PKL).exists():
    save_pickle(best_configs_l3_recall, str(config.BEST_CONFIGS_L3_RECALL_PKL))
    save_pickle(raw_l3_recall, str(config.RAW_L3_RECALL_PKL))
else:
    print(f'Pre-computed — pickles already exist, skipping saves')

In [ ]:
plot_smoothed_recall([raw_l3_recall, best_configs_l3_recall], 
                     ['Original', 'INF (best config per decile)'], 
                     'Mean Cross-Validation TPR at Different Levels of\nTarget Safety Knowledge (L3 Metapaths)', 
                     'Decile of Validation Target-AE Triples by No. of AEs Per Target\n(Decile Boundaries; Mean Across 5 CV Folds)', 'Mean CV TPR')

## 2. L-4

In [17]:
folds_l4_inf_feats_combos = load_pickle(str(config.CV_INF_FEATS_L34_PKL))

In [ ]:
from pathlib import Path as _Path
if _Path(config.CV_GS_RESULTS_L4_PKL).exists():
    inf_cv_gs_results_l4 = load_pickle(str(config.CV_GS_RESULTS_L4_PKL))
    print(f'Pre-computed — loaded {config.CV_GS_RESULTS_L4_PKL.name}')
else:
    inf_cv_gs_results_l4 = {}

    for combo_idx, combo in tqdm(enumerate(toolbox.param_combos), total=len(toolbox.param_combos)):

        inf_cv_gs_results_l4[combo_idx] = cv_gs_loop(folds_l4_inf_feats_combos[combo_idx], 'Gene_ADR_deg', ['Gene','ADR'])

In [19]:
if not (config.CV_GS_RESULTS_L4_PKL).exists():
    save_pickle(inf_cv_gs_results_l4, str(config.CV_GS_RESULTS_L4_PKL))
else:
    print(f'Pre-computed — {config.CV_GS_RESULTS_L4_PKL.name} already exists, skipping save')

#### Slice results output - recall

In [ ]:
# Raw counts slice results
analyse_parameter_slices_quantile(inf_cv_gs_results_l4[0][1])[['min_param','max_param',
                                                               'recall_mean', 'recall_ci',
                                                               'mean_pos_per_slice', 
                                                               'mean_true_pos', 'mean_pred_pos',
                                                               'precision_mean', 'precision_ci',
                                                               'f1_mean', 'f1_ci',
                                                               'mean_val_neg']]

In [ ]:
find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 1)[['min_param','max_param', # L4 is inferior so does not proceed to hits @ k
                                                               'recall_mean', 'recall_ci',
                                                               'mean_pos_per_slice', 
                                                               'mean_true_pos', 'mean_pred_pos',
                                                               'precision_mean', 'precision_ci',
                                                               'f1_mean', 'f1_ci',
                                                               'mean_val_neg']]

In [131]:
raw_l4_recall = analyse_parameter_slices_quantile(inf_cv_gs_results_l4[0][1])[['min_param', 'max_param', 'recall_mean', 'recall_ci']]

In [130]:
best_configs_l4_recall = find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 1)[['min_param', 'max_param', 'recall_mean', 'recall_ci']]

In [1568]:
if not (config.RAW_L4_RECALL_PKL).exists():
    save_pickle(raw_l4_recall, str(config.RAW_L4_RECALL_PKL))
    save_pickle(best_configs_l4_recall, str(config.BEST_CONFIGS_L4_RECALL_PKL))
else:
    print(f'Pre-computed — pickles already exist, skipping saves')

In [ ]:
plot_smoothed_recall([best_configs_l3_recall, best_configs_l4_recall], 
                     ['INF L-3 (best config per decile)', 'INF L-4 (best config per decile)'], 
                     'Mean Cross-Validation TPR at Different Levels of Target\nSafety Knowledge: INF Length-3 vs Length-4 Metapaths', 
                     'Decile of Validation Target-AE Triples by No. of AEs Per Target\n(Mean Decile Boundaries Across 5 CV Folds)', 'Mean CV TPR')

In [ ]:
from pathlib import Path as _Path
if _Path(config.CV_GS_RESULTS_L4_F1_PKL).exists():
    inf_cv_gs_results_l4_f1 = load_pickle(str(config.CV_GS_RESULTS_L4_F1_PKL))
    print(f'Pre-computed — loaded {config.CV_GS_RESULTS_L4_F1_PKL.name}')
else:
    # F1
    inf_cv_gs_results_l4_f1 = {}

    for combo_idx, combo in tqdm(enumerate(toolbox.param_combos), total=len(toolbox.param_combos)):

        inf_cv_gs_results_l4_f1[combo_idx] = cv_gs_loop(folds_l4_inf_feats_combos[combo_idx], 'Gene_ADR_deg', ['Gene','ADR'], 'f1')

In [ ]:
if not (config.CV_GS_RESULTS_L4_F1_PKL).exists():
    save_pickle(inf_cv_gs_results_l4_f1, str(config.CV_GS_RESULTS_L4_F1_PKL))
else:
    print(f'Pre-computed — {config.CV_GS_RESULTS_L4_F1_PKL.name} already exists, skipping save')

## 3. DWPC CV (single-config by design)

### Get CV TPR results

In [31]:
folds_dwpc_feats = load_pickle(str(config.CV_DWPC_FEATS_PKL))

In [38]:
from pathlib import Path as _Path
if _Path(config.CV_GS_RESULTS_L3_DWPC_PKL).exists():
    dwpc_cv_gs_results_l3_recall = load_pickle(str(config.CV_GS_RESULTS_L3_DWPC_PKL))
    print(f'Pre-computed — loaded {config.CV_GS_RESULTS_L3_DWPC_PKL.name}')
else:
    dwpc_cv_gs_results_l3_recall = cv_gs_loop(folds_dwpc_feats, 'Gene_ADR_deg', ['Gene','ADR'])

In [49]:
if not (config.CV_GS_RESULTS_L3_DWPC_PKL).exists():
    save_pickle(dwpc_cv_gs_results_l3_recall, str(config.CV_GS_RESULTS_L3_DWPC_PKL))
else:
    print(f'Pre-computed — {config.CV_GS_RESULTS_L3_DWPC_PKL.name} already exists, skipping save')

In [ ]:
analyse_parameter_slices_quantile(dwpc_cv_gs_results_l3_recall[1])[['min_param','max_param',
                                                                     'recall_mean', 'recall_ci',
                                                                     'mean_pos_per_slice', 
                                                                     'mean_true_pos', 'mean_pred_pos',
                                                                     'precision_mean', 'precision_ci',
                                                                     'f1_mean', 'f1_ci',
                                                                     'mean_val_neg']]

In [110]:
dwpc_l3_recall = analyse_parameter_slices_quantile(dwpc_cv_gs_results_l3_recall[1])[['min_param','max_param',
                                                                     'recall_mean', 'recall_ci']]

## CV Comparison Curve Plots

In [ ]:
plot_smoothed_recall([raw_l3_recall, best_configs_l3_recall], 
                     ['Original', "INF (best config for decile\n out of 34 eval'd)"], 
                     'Mean Cross-Validation TPR at Different Levels of\nTarget Safety Knowledge (L3 Metapaths)', 
                     'Decile of Validation Target-AE Triples by No. of AEs Per Target\n(Decile Boundaries; Mean Across 5 CV Folds)', 'Mean CV TPR')

In [ ]:
plot_smoothed_recall([dwpc_l3_recall, best_configs_l3_recall], 
                     ['DWPC (config as-is in source)', "INF (best config for decile)"], 
                     'Mean Cross-Validation TPR of Target-AE Triples\nat Different Levels of Target Safety Knowledge (L3 Metapaths)', 
                     'Decile of Target-AE Validation Triples by No. of AEs Per Target\n(Mean Decile Boundaries Across 5 CV Folds)', 'Mean CV TPR')

In [ ]:
plot_smoothed_recall([raw_l4_recall, best_configs_l4_recall], 
                     ['Original Values', "INF (best config for decile)"], 
                     'Mean Cross-Validation TPR at Different Levels of\nTarget Safety Knowledge (L4 Metapaths)', 
                     'Decile of Validation Target-AE Triples by No. of AEs Per Target\n(Mean Decile Boundaries Across 5 CV Folds)', 'Mean CV TPR')

# CV Charts to be Included

In [562]:
def plot_tpr_comparison(s1_tpr_values, s2_tpr_values, 
                        s1_ci_values, s2_ci_values,
                        feature_set_names, subset_names,
                        title, ylabel='Mean TPR', xlabel=None,
                        save_path=None):
    """
    Generates a visually appealing grouped bar chart to compare TPR values.

    Args:
        s1_tpr_values (list): Mean TPRs for the first feature set [subset1, subset2].
        s2_tpr_values (list): Mean TPRs for the second feature set [subset1, subset2].
        s1_ci_values (list): CI margins of error for the first feature set.
        s2_ci_values (list): CI margins of error for the second feature set.
        feature_set_names (list): Names for the two feature sets (e.g., ['S1', 'S2']).
        subset_names (list): Names for the data subsets (e.g., ['Subset 1', 'Subset 2']).
        title (str): The main title for the chart.
        ylabel (str): The label for the y-axis.
        xlabel (str, optional): The label for the x-axis. Defaults to None.
    """
    x = np.arange(len(subset_names))  # the label locations
    width = 0.35  # the width of the bars

    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create bars for the first feature set
    rects1 = ax.bar(x - width/2, s1_tpr_values, width, 
                    label=feature_set_names[0], 
                    yerr=s1_ci_values, capsize=5,
                    color='mediumseagreen')
    
    # Create bars for the second feature set
    rects2 = ax.bar(x + width/2, s2_tpr_values, width, 
                    label=feature_set_names[1],
                    yerr=s2_ci_values, capsize=5,
                    color='teal')

    # Add text labels, title, and custom x-axis tick labels, etc.
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_title(title, fontsize=16, pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(subset_names, fontsize=14)
    ax.legend(fontsize=11)
    ax.set_ylim(0, max(max(s1_tpr_values), max(s2_tpr_values)) + 0.2) # Dynamic y-limit
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Add numerical labels on top of each bar
    ax.bar_label(rects1, padding=3, fmt='%.2f')
    ax.bar_label(rects2, padding=3, fmt='%.2f')

    fig.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

### (1)	Comparison of TPR achieved on the 1st and 10th deciles (mean TPR for these deciles across all CV folds) by:
	the original count values of L3
	the original count values of L4
#### The comparison demonstrates the effects of topological imbalance on models using each metapath feature set.

In [ ]:
plot_tpr_comparison([0.202, 0.56, 0.959], [0.223, 0.575, 0.966], [0.071, 0.109, 0.031], [0.086, 0.09, 0.027],
                    ['L3 Original Values', 'L4 Original Values'], ['1st Decile (0-5 AEs)*', '2nd Decile (6-17 AEs)*', '10th Decile (163-256 AEs)*'],
                    save_path=str(config.FIGS_DIR / 'fig6_cv_tpr.png'),
                    title='Mean Cross-Validation TPR on the 2 Lowest Deciles vs the Highest Decile\nby No. of Known AEs for Target', ylabel='TPR (CV Mean)', xlabel=None)

In [585]:
all_cv_gs_results_l3 = copy.deepcopy(inf_cv_gs_results_l3)
all_cv_gs_results_l3['dwpc'] = dwpc_cv_gs_results_l3_recall

In [586]:
def ci_margin_std_given(stdev, n=5, confidence=0.95):
    """CI margin given a pre-computed standard deviation and number of observations."""
    if stdev == 0 or n < 2:
        return 0.0
    t_score = t.ppf((1 + confidence) / 2, df=n - 1)
    return t_score * stdev / np.sqrt(n)


def rank_config_results(config_results):
    """
    Takes a dictionary of configuration results and returns a DataFrame ranked
    by mean recall.

    Args:
        config_results (dict): A dictionary where each value is a tuple containing
                               (final_metrics, ...), and final_metrics is a list
                               or tuple like (recall_mean, stdev, ...).

    Returns:
        pd.DataFrame: A DataFrame with the results ranked by recall_mean.
    """
    results_list = []
    
    # 1. Iterate through each configuration's results
    for result_key, data_tuple in config_results.items():
        
        # 2. Extract the mean recall and standard deviation
        final_metrics = data_tuple[0]
        recall_mean = final_metrics[0]
        stdev = final_metrics[1]
        
        # 3. Calculate the CI margin using the provided function
        recall_ci = ci_margin_std_given(stdev)
        
        results_list.append({
            'result_key': result_key,
            'recall_mean': recall_mean,
            'recall_ci': recall_ci
        })
        
    # 4. Create a DataFrame and rank it by mean recall
    results_df = pd.DataFrame(results_list)
    ranked_df = results_df.sort_values('recall_mean', ascending=False).reset_index(drop=True)
    
    return ranked_df

In [588]:
ranked_config_results_l3 = rank_config_results(all_cv_gs_results_l3)

In [ ]:
ranked_config_results_l3_comparison = ranked_config_results_l3.iloc[:3]
ranked_config_results_l3_comparison = pd.concat([ranked_config_results_l3_comparison,
                                                 ranked_config_results_l3.loc[ranked_config_results_l3['result_key']=='dwpc'],
                                                 ranked_config_results_l3.loc[ranked_config_results_l3['result_key']==0
                                                                              ]]).reset_index(drop=True)
ranked_config_results_l3_comparison

In [ ]:
ranked_config_results_l3_comparison['Method'] = ['INF_1', 'INF_2', 'INF_3', 'DWPC', 'Original']

ranked_config_results_l3_comparison

In [657]:
def plot_ranked_bar_chart(results_df, title, value_col='recall_mean', ci_col='recall_ci'):
    """
    Generates a single horizontal bar chart showing a ranked list of methods
    from a dictionary of results.

    Args:
        results_dict (dict): A dictionary where keys are the method names and values
                             are pandas Series (or dicts) containing the scores.
        title (str): The title for the chart.
        value_col (str): The key/name for the mean score in the Series.
        ci_col (str): The key/name for the CI margin of error in the Series.
    """

    sorted_df = results_df.sort_values(value_col, ascending=True)
    
    # 3. Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.get_cmap('winter', len(sorted_df))

    # Create the horizontal bar plot with error bars
    bars = ax.barh(sorted_df['Method'], sorted_df[value_col], 
                   xerr=sorted_df[ci_col], capsize=5,
                   color=colors(np.linspace(0.25, 0.9, len(sorted_df))))
    
    # 4. Format the chart
    ax.set_title(title, fontsize=16)
    ax.set_xlabel('TPR (CV Mean)', fontsize=14)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_xlim(left=0.6, right=sorted_df[value_col].max() * 1.15)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_ranked_bar_chart(ranked_config_results_l3_comparison, 'TPR (All Validation Data, Mean across CV), L3 Metapaths:\nOriginal Values, DWPC & Top 3 INF Transformations')

In [662]:
ranked_config_results_l4 = rank_config_results(inf_cv_gs_results_l4)

In [ ]:
ranked_config_results_l4_comparison = ranked_config_results_l4.iloc[:3]
ranked_config_results_l4_comparison = pd.concat([ranked_config_results_l4_comparison,
                                                 ranked_config_results_l4.loc[ranked_config_results_l4['result_key']==0
                                                                              ]]).reset_index(drop=True)
ranked_config_results_l4_comparison

In [ ]:
ranked_config_results_l4_comparison['Method'] = ['INF_1', 'INF_2', 'INF_3', 'Original']

ranked_config_results_l4_comparison

In [ ]:
plot_ranked_bar_chart(ranked_config_results_l4_comparison, 'TPR (All Validation Data, Mean across CV), L4 Metapaths:\nOriginal Values & Top 3 INF Transformations')

## Bottom Deciles INF Best v benchmarks

In [ ]:
print('Decile 1 L3:')
print(f'Ranking:\n')

for i in range(1,6):

    dec1_best_configs_at_rank_i = find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', i).iloc[0]
    combo_idx = int(dec1_best_configs_at_rank_i['param_combo'])
    combo_details = param_combos_load[int(dec1_best_configs_at_rank_i['param_combo'])]

    print(f'{i}: Param combo {combo_idx} -- {combo_details}')
    print(dec1_best_configs_at_rank_i[['recall_mean', 'recall_ci']])
    print('\n')


## 1st Decile INF Best v benchmarks

In [ ]:
dec1_l3_best_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 1).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 2).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 3).iloc[[0]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1]).iloc[[0]],
                                  analyse_parameter_slices_quantile(dwpc_cv_gs_results_l3_recall[1]).iloc[[0]]])


dec1_l3_best_configs

In [ ]:
dec1_l3_best_configs['Method'] = ['INF_1', 'INF_2', 'INF_3', 'Original', 'DWPC']

dec1_l3_best_configs

In [ ]:
dec1_l3_best_configs[['Method', 'recall_mean', 'recall_ci']]

In [677]:
def plot_ranked_bar_chart_2(results_df, title, value_col='recall_mean', ci_col='recall_ci'):
    """
    Generates a single horizontal bar chart showing a ranked list of methods
    from a dictionary of results.

    Args:
        results_dict (dict): A dictionary where keys are the method names and values
                             are pandas Series (or dicts) containing the scores.
        title (str): The title for the chart.
        value_col (str): The key/name for the mean score in the Series.
        ci_col (str): The key/name for the CI margin of error in the Series.
    """

    sorted_df = results_df.sort_values(value_col, ascending=True)
    
    # 3. Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.get_cmap('winter', len(sorted_df))

    # Create the horizontal bar plot with error bars
    bars = ax.barh(sorted_df['Method'], sorted_df[value_col], 
                   xerr=sorted_df[ci_col], capsize=5,
                   color=colors(np.linspace(0.25, 0.9, len(sorted_df))))
    
    # 4. Format the chart
    ax.set_title(title, fontsize=16)
    ax.set_xlabel('TPR (CV Mean)', fontsize=14)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_xlim(left=0, right=sorted_df[value_col].max() * 1.5)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_ranked_bar_chart_2(dec1_l3_best_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 1st Decile of Target-AE Assoc's\nby No. of Known AEs for Target (0-5 AEs); L3 Metapaths")

In [ ]:
dec1_l4_best_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 1).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 2).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 3).iloc[[0]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l4[0][1]).iloc[[0]]])

dec1_l4_best_configs

In [ ]:
dec1_l4_best_configs['Method'] = ['INF_1', 'INF_2', 'INF_3', 'Original']

dec1_l4_best_configs

In [ ]:
plot_ranked_bar_chart_2(dec1_l4_best_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 1st Decile of Target-AE Assoc's\nby No. of Known AEs for Target (0-5 AEs); L4 Metapaths")

## Verdict 1st Decile best - 0 deflator is pervasive, but so is INF term. INF exceeds benchmarks.

## 1st Decile INF Worst v benchmarks

In [ ]:
dec1_l3_worst_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 34).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 33).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 32).iloc[[0]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1]).iloc[[0]],
                                  analyse_parameter_slices_quantile(dwpc_cv_gs_results_l3_recall[1]).iloc[[0]]])


dec1_l3_worst_configs

In [ ]:
dec1_l3_worst_configs['Method'] = ['INF_34', 'INF_33', 'INF_32', 'Original', 'DWPC']

dec1_l3_worst_configs

In [ ]:
plot_ranked_bar_chart_2(dec1_l3_worst_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 1st Decile of Target-AE Assoc's\nby No. of Known AEs for Target (0-5 AEs); L3 Metapaths")

In [ ]:
dec1_l4_worst_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 35).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 34).iloc[[0]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 33).iloc[[0]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l4[0][1]).iloc[[0]]])

dec1_l4_worst_configs

In [ ]:
dec1_l4_worst_configs['Method'] = ['INF_34', 'INF_33', 'INF_32', 'Original']

dec1_l4_worst_configs

In [ ]:
plot_ranked_bar_chart_2(dec1_l4_worst_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 1st Decile of Target-AE Assoc's\nby No. of Known AEs for Target (0-5 AEs); L4 Metapaths")

## Verdict 1st Decile Worst: 2/3 L3 sets don't have any INF term; 1/3 for L4. Where INF term present, min pooling is the option. 2/3 have 0 deflator in L4, 1/3 in L3. Suggests both path rarity and count have value for these data.
### Original hovers close to worst INF in both L3 and L4. DWPC is better but not by much.

## 2nd Decile INF Best v benchmarks

In [ ]:
dec2_l3_best_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 1).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 2).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 3).iloc[[1]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1]).iloc[[1]],
                                  analyse_parameter_slices_quantile(dwpc_cv_gs_results_l3_recall[1]).iloc[[1]]])


dec2_l3_best_configs

In [ ]:
dec2_l3_best_configs['Method'] = ['INF_1', 'INF_2', 'INF_3', 'Original', 'DWPC']

dec2_l3_best_configs

In [709]:
def plot_ranked_bar_chart_3(results_df, title, value_col='recall_mean', ci_col='recall_ci'):
    """
    Generates a single horizontal bar chart showing a ranked list of methods
    from a dictionary of results.

    Args:
        results_dict (dict): A dictionary where keys are the method names and values
                             are pandas Series (or dicts) containing the scores.
        title (str): The title for the chart.
        value_col (str): The key/name for the mean score in the Series.
        ci_col (str): The key/name for the CI margin of error in the Series.
    """

    sorted_df = results_df.sort_values(value_col, ascending=True)
    
    # 3. Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.get_cmap('winter', len(sorted_df))

    # Create the horizontal bar plot with error bars
    bars = ax.barh(sorted_df['Method'], sorted_df[value_col], 
                   xerr=sorted_df[ci_col], capsize=5,
                   color=colors(np.linspace(0.25, 0.9, len(sorted_df))))
    
    # 4. Format the chart
    ax.set_title(title, fontsize=16)
    ax.set_xlabel('TPR (CV Mean)', fontsize=14)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_xlim(left=0.2, right=sorted_df[value_col].max() * 1.5)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_ranked_bar_chart_3(dec2_l3_best_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 2nd Decile of Target-AE Assoc's\nby No. of Known AEs for Target (6-17 AEs); L3 Metapaths")

In [ ]:
dec2_l4_best_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 1).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 2).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 3).iloc[[1]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l4[0][1]).iloc[[1]]])

dec2_l4_best_configs

In [ ]:
dec2_l4_best_configs['Method'] = ['INF_1', 'INF_2', 'INF_3', 'Original']

dec2_l4_best_configs

In [ ]:
plot_ranked_bar_chart_3(dec2_l4_best_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 2nd Decile of Target-AE Assoc's\nby No. of Known AEs for Target (6-17 AEs); L4 Metapaths")

## Verdict 2nd Decile Best: 7 surprises in L3. Other 2 have INF. 2/3 don't have deflator in L4, but all L4 have INF. Best INF has clear lead over both benchmarks.

## 2nd Decile Deciles INF Worst v benchmarks

In [ ]:
dec2_l3_worst_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 35).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 34).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 33).iloc[[1]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1]).iloc[[1]],
                                  analyse_parameter_slices_quantile(dwpc_cv_gs_results_l3_recall[1]).iloc[[1]]])


dec2_l3_worst_configs

In [ ]:
dec2_l3_worst_configs['Method'] = ['INF_34', 'INF_33', 'INF_32', 'Original', 'DWPC']

dec2_l3_worst_configs

In [ ]:
plot_ranked_bar_chart_3(dec2_l3_worst_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 2nd Decile of Target-AE Assoc's\nby No. of Known AEs for Target (6-17 AEs); L3 Metapaths")

In [ ]:
dec2_l4_worst_configs = pd.concat([find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 35).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 34).iloc[[1]],
                                  find_best_configs_quantile(inf_cv_gs_results_l4, 'Gene_ADR_deg', 33).iloc[[1]],
                                  analyse_parameter_slices_quantile(inf_cv_gs_results_l4[0][1]).iloc[[1]]])

dec2_l4_worst_configs

In [ ]:
dec2_l4_worst_configs['Method'] = ['INF_34', 'INF_33', 'INF_32', 'Original']

dec2_l4_worst_configs

In [ ]:
plot_ranked_bar_chart_3(dec2_l4_worst_configs[['Method', 'recall_mean', 'recall_ci']].reset_index(), "Mean CV TPR, 2nd Decile of Target-AE Assoc's\nby No. of Known AEs for Target (6-17 AEs); L4 Metapaths")

## Verdict 2nd Decile Worst: Min pooling option + 0 deflator in 2/3 worst L3 and 3/3 L4 combinations. 7 among worst in L4. 
### Indicates discarding count data (0 deflator) while taking smallest boost from INF (min pooling) especially destructive - both count and node rarity important - decile seems to be data transition between path rarity being most important and count being most important.

# II. Testing with rank-based metrics

### Prepare INF data

In [52]:
inf_test_feats_dir = str(config.INF_FEATS_DIR)

In [ ]:

negs_to_test_pkl = load_pickle(f'{inf_test_feats_dir}/negs_to_test.pkl')

negs_to_test_pkl

In [251]:
negs_to_test = copy.deepcopy(negs_to_test_pkl)

In [ ]:
train_test_pkl = load_pickle(f'{inf_test_feats_dir}/train_test.pkl')
train_test_pkl.keys()

#### Verify INF values

#### Get high-degree example

In [262]:
inf_hitk_train_test_feats = {'train_pos': train_test_pkl['pos_train'],
                            'train_neg': train_test_pkl['neg_train'],
                            'negs_to_test': negs_to_test}

In [93]:
gene_match_adr_query = '''
MATCH (n:Gene)-[rel:Gene_ADR]->() WHERE n.name = $gene RETURN COUNT(rel) AS gene_adr_deg'''

In [ ]:
from pathlib import Path as _Path
if not _Path(config.HITK_PARTITIONS_PKL).exists():
    genes_degs_adr = {}
                    
    for gene in tqdm(all_genes_aes):

        genes_degs_adr[gene] = toolbox.graph.run(gene_match_adr_query, gene=gene).evaluate()

    print('AE per Gene counting (w/ test edges deleted) complete')

    genes_degs_adr_df = pd.Series(genes_degs_adr).reset_index().rename(columns={'index': 'Gene', 0: 'Gene_ADR_deg'})

    genes_degs_adr_df.fillna(0, inplace=True)

    assert not genes_degs_adr_df.isna().any().any(), 'Nans found!'
else:
    print(f'Pre-computed — skipping ({config.HITK_PARTITIONS_PKL.name} exists)')

In [ ]:
from pathlib import Path as _Path
if _Path(config.HITK_PARTITIONS_PKL).exists():
    inf_hitk_train_test_feats = load_pickle(str(config.HITK_PARTITIONS_PKL))
    print(f'Pre-computed — loaded {config.HITK_PARTITIONS_PKL.name}')
else:
    for partition_key in inf_hitk_train_test_feats:

        print(f'Doing {partition_key}...')

        for combo_idx, _ in tqdm(enumerate(toolbox.param_combos), total=len(toolbox.param_combos)):

            if 'Gene' and 'ADR' not in inf_hitk_train_test_feats[partition_key][combo_idx].columns:

                inf_hitk_train_test_feats[partition_key][combo_idx] = toolbox.restore_pair_node_id_cols(inf_hitk_train_test_feats[partition_key][combo_idx],
                                                                                                        'Gene', 0, 'ADR', 1)
        
            if 'Gene_ADR_deg' in inf_hitk_train_test_feats[partition_key][combo_idx].columns:

                inf_hitk_train_test_feats[partition_key][combo_idx] = inf_hitk_train_test_feats[partition_key][combo_idx].drop(columns=['Gene_ADR_deg'])
        
            inf_hitk_train_test_feats[partition_key][combo_idx] = pd.merge(inf_hitk_train_test_feats[partition_key][combo_idx],
                                                                                genes_degs_adr_df, how='left', on='Gene')

In [265]:
if not (config.HITK_PARTITIONS_PKL).exists():
    save_pickle(inf_hitk_train_test_feats, str(config.HITK_PARTITIONS_PKL))
else:
    print(f'Pre-computed — {config.HITK_PARTITIONS_PKL.name} already exists, skipping save')

### Function
##### Takes in pos & neg train, learns model and makes forward pass

In [ ]:
raw_l3_recall = analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1])[['min_param', 'max_param', 'recall_mean', 'recall_ci']]

In [ ]:
best_configs_l3 = find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 1)[['slice_num', 'param_combo', 'min_param', 'max_param', 'recall_mean', 'recall_ci']]
best_configs_l3

In [ ]:
from typing import List, Set, Tuple, Optional, Any

## Old Hit K formulation (good but no CIs & uncorrected decile bounds)

In [79]:
def train_link_prediction_model(head_col: str, tail_col: str,
                                train_pos: pd.DataFrame,
                                train_neg: pd.DataFrame,
                                neg_to_test: pd.DataFrame,
                                test_pos_pairs: pd.DataFrame,
                                cols_to_drop: Optional[List[str]] = None,
                                **rf_kwargs: Any
) -> Tuple[RandomForestClassifier, pd.DataFrame]:
    """
    Trains a Random Forest model for link prediction and prepares the test set.

    Args:
        train_pos (pd.DataFrame): DataFrame of positive training examples.
        train_neg (pd.DataFrame): DataFrame of negative training examples.
        neg_to_test (pd.DataFrame): DataFrame of candidate pairs for evaluation.
        test_pos_pairs (pd.DataFrame): A DataFrame with 'head' and 'tail' columns
                                       representing the ground-truth positive pairs.
        cols_to_drop (Optional[List[str]], optional): Additional feature
            columns to drop before training. Defaults to None.
        **rf_kwargs: Optional keyword arguments to pass to the
                     RandomForestClassifier (e.g., max_depth, n_jobs).

    Returns:
        Tuple[RandomForestClassifier, pd.DataFrame]: A tuple containing:
            - The trained scikit-learn model object.
            - The neg_to_test DataFrame, now with 'true_label' and
              'model_score' columns, ready for evaluation.
    """
    # --- 1. Prepare and Check the Test Data ---
    # Convert the ground-truth DataFrame to a set of tuples for efficient lookup
    ground_truth_set = set(zip(test_pos_pairs[head_col], test_pos_pairs[tail_col]))

    # Create a set of pairs from the test candidates
    test_candidate_pairs = set(zip(neg_to_test[head_col], neg_to_test[tail_col]))

    # Check if all ground-truth pairs are present in the candidate set
    if not ground_truth_set.issubset(test_candidate_pairs):
        missing_pairs = len(ground_truth_set - test_candidate_pairs)
        raise ValueError(
            f"{missing_pairs} pairs from 'test_pos_pairs' are missing "
            "from the 'neg_to_test' candidate DataFrame."
        )

    # --- 2. Prepare Training Data ---
    train_pos = train_pos.assign(label=1)
    train_neg = train_neg.assign(label=0)
    train_df = pd.concat([train_pos, train_neg], ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

    final_cols_to_drop = [head_col, tail_col] + (cols_to_drop or [])
    feature_cols = [col for col in train_df.columns if col not in final_cols_to_drop + ['label']]
    X_train = train_df[feature_cols]
    y_train = train_df['label']

    # --- 3. Train the Random Forest Model ---
    model = RandomForestClassifier(n_estimators=200, random_state=69, **rf_kwargs)
    print("Training Random Forest model...")
    model.fit(X_train, y_train)
    print("Training complete.")

    # --- 4. Prepare Test Set and Predict Scores ---
    eval_df = neg_to_test.copy()
    eval_df['true_label'] = eval_df.apply(
        lambda row: (row[head_col], row[tail_col]) in ground_truth_set,
        axis=1
    )
    X_test = eval_df[feature_cols]
    prob_scores = model.predict_proba(X_test)[:, 1]
    eval_df['model_score'] = prob_scores

    return model, eval_df[[head_col, tail_col, 'true_label', 'model_score']]

### INF

In [76]:
test_pos_pairs_load = load_pickle(str(config.TEST_POS_PAIRS_PKL))

In [ ]:
from pathlib import Path as _Path
if _Path(config.HITK_DFS_PKL).exists():
    hitk_dfs = load_pickle(str(config.HITK_DFS_PKL))
    print(f'Pre-computed — loaded {config.HITK_DFS_PKL.name}')
else:
    hitk_dfs = {combo_idx: None for combo_idx, _ in enumerate(toolbox.param_combos)}

    for combo_idx, combo in enumerate(toolbox.param_combos):

        print(f'Now preparing combo {combo_idx}: ({combo})')

        hitk_dfs[combo_idx] = train_link_prediction_model('Gene', 'ADR',
                                                          inf_hitk_train_test_feats['train_pos'][combo_idx],
                                                          inf_hitk_train_test_feats['train_neg'][combo_idx],
                                                          inf_hitk_train_test_feats['negs_to_test'][combo_idx],
                                                          test_pos_pairs_load,
                                                          ['Gene_ADR_deg'],
                                                          **inf_cv_gs_results_l3[combo_idx][2])

### DWPC

In [84]:
dwpc_negs_to_test_dir = str(config.DWPC_FEATS_DIR)
dwpc_negs_to_test = load_pickle(f'{dwpc_negs_to_test_dir}/negs_to_test_l3_feats_ids.pkl')

In [85]:
dwpc_pos_train = load_pickle(f'{dwpc_negs_to_test_dir}/train_pos_dwpc.pkl')
dwpc_neg_train = load_pickle(f'{dwpc_negs_to_test_dir}/train_neg_dwpc.pkl')

In [86]:
dwpc_pos_train.drop(columns=['Label', 'PairID'], inplace=True)

In [87]:
dwpc_neg_train.drop(columns=['Label', 'PairID'], inplace=True)

In [ ]:
dwpc_negs_to_test.pop('Label')

In [ ]:
len(dwpc_pos_train.columns) == len(dwpc_neg_train.columns) == len(dwpc_negs_to_test.columns)

In [90]:
dwpc_hitk_train_test_feats = {'train_pos': dwpc_pos_train,
                            'train_neg': dwpc_neg_train,
                            'negs_to_test': dwpc_negs_to_test}

In [ ]:
# Build gene degree lookup from the pre-computed INF partition (already has Gene_ADR_deg)
genes_degs_adr_df = inf_hitk_train_test_feats['negs_to_test'][0][['Gene', 'Gene_ADR_deg']].drop_duplicates('Gene')

for partition_key in dwpc_hitk_train_test_feats:
    if 'Gene_ADR_deg' in dwpc_hitk_train_test_feats[partition_key].columns:
        dwpc_hitk_train_test_feats[partition_key] = dwpc_hitk_train_test_feats[partition_key].drop(columns=['Gene_ADR_deg'])
    dwpc_hitk_train_test_feats[partition_key] = pd.merge(
        dwpc_hitk_train_test_feats[partition_key], genes_degs_adr_df, how='left', on='Gene')
    print(f'{partition_key}: Gene_ADR_deg merged in')

In [100]:
dwpc_cv_gs_results_l3_recall = load_pickle(str(config.CV_GS_RESULTS_L3_DWPC_PKL))

In [ ]:
from pathlib import Path as _Path
if not _Path(config.HITK_DFS_PKL).exists():
    hitk_dfs['dwpc'] = train_link_prediction_model('Gene', 'ADR',
                                                          dwpc_hitk_train_test_feats['train_pos'],
                                                          dwpc_hitk_train_test_feats['train_neg'],
                                                          dwpc_hitk_train_test_feats['negs_to_test'],
                                                          test_pos_pairs_load,
                                                          ['Gene_ADR_deg'],
                                                          **dwpc_cv_gs_results_l3_recall[2])
else:
    print(f'Pre-computed — skipping ({config.HITK_DFS_PKL.name} exists)')

### Get overall & slice ranked metrics for all configs

In [103]:
def filtered_evaluation(df: pd.DataFrame, head_col: str, tail_col: str, k: int = 10) -> Dict[str, float]:
    """
    Calculates filtered Hits@k and overall classic recall.

    Args:
        df (pd.DataFrame): A DataFrame with columns ['head', 'tail', 
                          'true_label', 'model_score']. It should contain all
                          candidate tails for each head.
        head_col (str): The name of the head entity column.
        tail_col (str): The name of the tail entity column.
        k (int, optional): The threshold for Hits@k. Defaults to 10.

    Returns:
        dict: A dictionary containing the calculated metrics:
              Hits@k and Recall.
    """
    
    all_filtered_ranks = []
    
    # Track overall true positives and all actual positives for overall recall calculation
    total_true_positives = 0
    total_actual_positives = 0

    # Group by head entity to evaluate each link prediction task separately
    for head, group in df.groupby(head_col):
        
        # Sort candidates by model score in descending order
        sorted_group = group.sort_values('model_score', ascending=False)
        
        # Get the set of all true tails for this head
        true_tails = set(sorted_group[sorted_group['true_label'] == True][tail_col])
        
        # Count total actual positives (for overall recall)
        total_actual_positives += len(true_tails)
        
        if not true_tails:
            continue

        # --- Hits@k (filtered) Calculation ---
        # Calculate a filtered rank for each true tail
        ranked_tails = sorted_group[tail_col].tolist()
        
        for true_t in true_tails:
            try:
                raw_rank_idx = ranked_tails.index(true_t)
            except ValueError:
                continue
            
            num_other_true_tails_ranked_higher = len([
                t for t in ranked_tails[:raw_rank_idx] if t in true_tails
            ])
            
            filtered_rank = (raw_rank_idx + 1) - num_other_true_tails_ranked_higher
            all_filtered_ranks.append(filtered_rank)

        # --- Overall Classic Recall Calculation ---
        # A positive prediction is a score >= 0.5 (scikit-learn default)
        predicted_positives = set(sorted_group[sorted_group['model_score'] >= 0.5][tail_col])
        
        # True Positives for this head
        true_positives_in_head = len(predicted_positives.intersection(true_tails))
        
        # Add to the running total
        total_true_positives += true_positives_in_head

    # --- Aggregate the final metrics ---
    metrics = {}
    
    # Calculate Hits@k
    all_filtered_ranks = np.array(all_filtered_ranks)
    if all_filtered_ranks.size > 0:
        hits_at_k = np.mean(all_filtered_ranks <= k)
        metrics[f'Hits@{k}'] = hits_at_k
    else:
        metrics[f'Hits@{k}'] = 0.0

    # Calculate overall recall
    if total_actual_positives > 0:
        overall_recall = total_true_positives / total_actual_positives
        metrics['Recall'] = overall_recall
    else:
        metrics['Recall'] = 0.0
        
    return metrics

In [ ]:
test_atk_metrics = {cfg_key: None for cfg_key in hitk_dfs} # default k is 10

for cfg_key in tqdm(hitk_dfs):

    test_atk_metrics[cfg_key] = filtered_evaluation(hitk_dfs[cfg_key][1], 'Gene', 'ADR')

In [ ]:
test_at20_metrics = {cfg_key: None for cfg_key in hitk_dfs}

for cfg_key in tqdm(hitk_dfs):

    test_at20_metrics[cfg_key] = filtered_evaluation(hitk_dfs[cfg_key][1], 'Gene', 'ADR', 20)

#### Re-add degs

In [ ]:
for cfg_key in tqdm(hitk_dfs):

    hitk_dfs[cfg_key][1]['Gene_ADR_deg'] = inf_hitk_train_test_feats['negs_to_test'][0]['Gene_ADR_deg']

In [109]:
if not (config.HITK_DFS_PKL).exists():
    save_pickle(hitk_dfs, str(config.HITK_DFS_PKL))
else:
    print(f'Pre-computed — {config.HITK_DFS_PKL.name} already exists, skipping save')

## Get per-slice ranking metrics

In [108]:
def compute_sliced_metrics(
    head_col, tail_col,
    boundaries_df: pd.DataFrame,
    eval_df: pd.DataFrame,
    param_col_name: str,
    filtered_eval_func: callable,
    k: int = 10
) -> pd.DataFrame:
    """
    Computes evaluation metrics on slices of data defined by parameter boundaries.

    It handles non-contiguous boundaries by setting the new boundary at the
    midpoint of the gap between slices.

    Args:
        boundaries_df (pd.DataFrame): DataFrame with '# min_param' and
                                      '# max_param' columns defining the slices.
        eval_df (pd.DataFrame): The evaluation data, containing the parameter
                                column to slice on and model scores.
        param_col_name (str): The name of the column in eval_df to slice by.
        filtered_eval_func (callable): The function to compute metrics
                                       (e.g., the one provided previously).
        k (int, optional): The 'k' value for the evaluation metrics. Defaults to 10.

    Returns:
        pd.DataFrame: A DataFrame with metrics for each parameter slice.
    """
    if boundaries_df.empty:
        return pd.DataFrame()

    # --- 1. Adjust boundaries to handle non-contiguous ranges ---
    # Calculate the midpoint between the max of one row and the min of the next
    next_min = boundaries_df['min_param'].shift(-1)
    midpoints = (boundaries_df['max_param'] + next_min) / 2.0

    # Create new adjusted min and max boundary Series
    # The new 'max' for a slice is the midpoint between it and the next slice
    # The new 'min' is the midpoint between it and the previous slice
    adjusted_max = midpoints.fillna(boundaries_df['max_param'])
    adjusted_min = midpoints.shift(1).fillna(boundaries_df['min_param'])

    # --- 2. Iterate through slices, compute metrics, and store results ---
    results_list = []
    for idx, row in boundaries_df.iterrows():
        min_bound = adjusted_min.loc[idx]
        max_bound = adjusted_max.loc[idx]

        # Define the slice (inclusive of boundaries)
        # Note: The first slice's min and last slice's max use original values
        if idx == 0: # First slice
             data_slice = eval_df[eval_df[param_col_name] <= max_bound]
        else:
             data_slice = eval_df[
                 (eval_df[param_col_name] > min_bound) &
                 (eval_df[param_col_name] <= max_bound)
             ]

        # Prepare the record for this slice
        slice_result = {
            'slice_idx': idx,
            'min_param': row['min_param'],
            'max_param': row['max_param']
        }

        # If the slice has data, compute metrics, otherwise fill with NaN
        if not data_slice.empty:
            metrics = filtered_eval_func(data_slice, head_col, tail_col, k=k)
            slice_result.update(metrics)
        else:
            # If you want to see which metrics would be calculated
            # You can create a dummy dict with NaN values
            # This is optional but makes the output shape consistent
            dummy_metrics = {
                'Mean Rank': np.nan, 'MRR': np.nan, f'Hits@{k}': np.nan,
                f'Mean Precision@{k}': np.nan, f'Mean Recall@{k}': np.nan
            }
            slice_result.update(dummy_metrics)

        results_list.append(slice_result)

    return pd.DataFrame(results_list)

In [ ]:
bounds_data = {
    'min_param': [0, 6],
    'max_param': [5, 17]
}
ae_deg_boundaries = pd.DataFrame(bounds_data)

In [ ]:
per_slice_atk_metrics = {cfg_key: None for cfg_key in hitk_dfs} # fot k = 10

for cfg_key in tqdm(hitk_dfs):
    
    per_slice_atk_metrics[cfg_key] = compute_sliced_metrics('Gene', 'ADR',
                                                           ae_deg_boundaries,
                                                           hitk_dfs[cfg_key][1],
                                                           'Gene_ADR_deg',
                                                           filtered_evaluation)

In [ ]:
per_slice_at20_metrics = {cfg_key: None for cfg_key in hitk_dfs}

for cfg_key in tqdm(hitk_dfs):
    
    per_slice_at20_metrics[cfg_key] = compute_sliced_metrics('Gene', 'ADR',
                                                           ae_deg_boundaries,
                                                           hitk_dfs[cfg_key][1],
                                                           'Gene_ADR_deg', filtered_evaluation, 20)

In [ ]:
def get_best_config( #overall; use with hits @ k only as ranking order is wrong
    config_metrics: dict,
    metric_name: str
) -> str:
    """
    Finds the config key with the highest value for a selected metric.

    Args:
        config_metrics (dict): A dictionary where keys are config names
                               and values are dictionaries of metrics.
                               Example: {'config1': {'MRR': 0.8, 'Hits@10': 0.9}}
        metric_name (str): The name of the metric to optimize (e.g., 'MRR').

    Returns:
        str: The key of the configuration with the highest metric score.
    """
    if not config_metrics:
        return None

    return max(
        config_metrics,
        key=lambda k: config_metrics[k].get(metric_name, -1)
    )

In [74]:
from typing import Dict, List # rank configs by metric on negs_to_test slice

def rank_configs_by_slice_metric(
    configs_results: Dict[str, pd.DataFrame],
    metric_name: str,
    slice_idx: int,
    top_k: int = 3
) -> List[str]:
    """
    Ranks configurations based on a chosen metric from a specific slice.

    Args:
        configs_results (Dict[str, pd.DataFrame]): A dictionary where keys are
            config names and values are the per-slice metrics DataFrames.
        metric_name (str): The name of the metric to rank by (e.g., 'MRR', 'Mean Rank').
        slice_idx (int): The index of the slice to use for ranking.
        top_k (int, optional): The number of top configurations to return. Defaults to 3.

    Returns:
        List[str]: A list of the top-k configuration keys, ordered from best to worst.
    """
    if not configs_results:
        return []

    # --- 1. Define which metrics are better when lower ---
    lower_is_better = {'Mean Rank'}
    sort_ascending = metric_name in lower_is_better

    # --- 2. Extract the specific slice score for each configuration ---
    slice_scores = {}
    for config_name, metrics_df in configs_results.items():
        # Default to a "worst-possible" score
        score = -np.inf if not sort_ascending else np.inf

        # Check that the required columns exist
        if metric_name not in metrics_df.columns or 'slice_idx' not in metrics_df.columns:
            print(f"Warning: Columns not found for '{config_name}'. Skipping.")
            slice_scores[config_name] = score
            continue

        # Find the row for the specified slice
        target_row = metrics_df[metrics_df['slice_idx'] == slice_idx]

        # If the slice exists and has a valid score, extract it
        if not target_row.empty:
            value = target_row.iloc[0][metric_name]
            if not pd.isna(value):
                score = value
        
        slice_scores[config_name] = score

    # --- 3. Sort configurations by the slice score and return the top k ---
    sorted_configs = sorted(
        slice_scores.keys(),
        key=lambda k: slice_scores[k],
        reverse=not sort_ascending
    )

    return sorted_configs[:top_k]

### Hits @ 10 slice-wise

### Slice 1 best

In [ ]:
for i, cfg_key in enumerate(rank_configs_by_slice_metric(per_slice_atk_metrics, 'Hits@10', 0, 5)):

    print(i+1, cfg_key, per_slice_atk_metrics[cfg_key]['Hits@10'].iloc[0], param_combos_load[cfg_key])

### Slice 2 best

### Slice 10 best

### Hits @ 20 slice-wise

### Slice 1 best

### Slice 2 best

### Slice 10 best

## New Hits K (for CIs and proper bounds)

### Setup

In [ ]:
inf_cv_gs_results_l3 = load_pickle(str(config.CV_GS_RESULTS_L3_PKL))
inf_cv_gs_results_l3[0]

In [40]:
# (find_best_configs_quantile defined earlier)

In [ ]:
best_configs_l3 = find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 1)[['slice_num', 'param_combo', 'min_param', 'max_param', 'recall_mean', 'recall_ci']]
best_configs_l3

In [50]:
if not (config.BEST_CONFIGS_L3_RECALL_PKL).exists():
    save_pickle(best_configs_l3, str(config.BEST_CONFIGS_L3_RECALL_PKL))
else:
    print(f'Pre-computed — {config.BEST_CONFIGS_L3_RECALL_PKL.name} already exists, skipping save')

In [63]:
def calculate_hits_at_k(df: pd.DataFrame, k: int, head_col, tail_col, n_bootstrap: int = 1000) -> Tuple[float, Tuple[float, float]]:
    """
    Calculates Hits@k and its 95% bootstrap confidence interval.
    """
    all_filtered_ranks = []
    
    if df.empty:
        return 0.0, (0.0, 0.0)
        
    for head, group in df.groupby(head_col):
        sorted_group = group.sort_values('model_score', ascending=False)
        true_tails = set(sorted_group[sorted_group['true_label'] == True][tail_col])
        
        if not true_tails:
            continue
            
        ranked_tails = sorted_group[tail_col].tolist()
        for true_t in true_tails:
            try:
                raw_rank_idx = ranked_tails.index(true_t)
            except ValueError:
                continue
                
            num_other_true_tails_ranked_higher = len([
                t for t in ranked_tails[:raw_rank_idx] if t in true_tails
            ])
            filtered_rank = (raw_rank_idx + 1) - num_other_true_tails_ranked_higher
            all_filtered_ranks.append(filtered_rank)
            
    filtered_ranks = np.array(all_filtered_ranks)
    
    if filtered_ranks.size == 0:
        return 0.0, (0.0, 0.0)
        
    hits_k_score = np.mean(filtered_ranks <= k)
    
    bootstrap_samples = np.random.choice(filtered_ranks, (n_bootstrap, len(filtered_ranks)), replace=True)
    bootstrap_means = np.mean(bootstrap_samples <= k, axis=1)
    ci_lower = np.percentile(bootstrap_means, 2.5)
    ci_upper = np.percentile(bootstrap_means, 97.5)
    
    return hits_k_score, (ci_lower, ci_upper)

In [65]:
def calculate_recall(df: pd.DataFrame, head_col: str, tail_col: str, n_bootstrap: int = 1000) -> Tuple[float, Tuple[float, float]]:
    """
    Calculates recall and its 95% bootstrap confidence interval.
    
    Recall is defined as: True Positives / (True Positives + False Negatives)
    A positive prediction is defined by a score >= 0.5.
    """
    all_positive_predictions = []
    
    if df.empty:
        return 0.0, (0.0, 0.0)
        
    for head, group in df.groupby(head_col):
        # Identify the true positive pairs for this head entity
        true_positives_in_group = group[group['true_label'] == True]
        
        # A positive prediction is where the model_score is >= 0.5
        predicted_positives_in_group = group[group['model_score'] >= 0.5]
        
        # True Positives (TP): A true label and a positive prediction
        true_positives = len(predicted_positives_in_group.merge(
            true_positives_in_group, 
            on=[head_col, tail_col], 
            how='inner'
        ))
        
        # False Negatives (FN): A true label but a negative prediction (score < 0.5)
        # These are the true positives that were not in the predicted positives
        false_negatives = len(true_positives_in_group) - true_positives
        
        # Store the individual results for this group
        if (true_positives + false_negatives) > 0:
            all_positive_predictions.append({
                'true_positives': true_positives,
                'false_negatives': false_negatives
            })

    if not all_positive_predictions:
        return 0.0, (0.0, 0.0)
    
    # Bootstrap for confidence interval
    recall_scores = []
    sample_indices = np.arange(len(all_positive_predictions))
    
    for _ in range(n_bootstrap):
        bootstrap_indices = np.random.choice(sample_indices, size=len(sample_indices), replace=True)
        bootstrap_sample = [all_positive_predictions[i] for i in bootstrap_indices]
        
        total_tp = sum(item['true_positives'] for item in bootstrap_sample)
        total_fn = sum(item['false_negatives'] for item in bootstrap_sample)
        
        if (total_tp + total_fn) > 0:
            bootstrap_recall = total_tp / (total_tp + total_fn)
            recall_scores.append(bootstrap_recall)
        else:
            recall_scores.append(0.0)

    # Calculate overall recall and confidence interval
    recall_score = np.mean(recall_scores)
    ci_lower = np.percentile(recall_scores, 2.5)
    ci_upper = np.percentile(recall_scores, 97.5)
    
    return recall_score, (ci_lower, ci_upper)

In [59]:
def train_and_predict_single_model(
    train_pos: pd.DataFrame, 
    train_neg: pd.DataFrame, 
    eval_candidates: pd.DataFrame, 
    test_pos_pairs: pd.DataFrame, 
    head_col: str, 
    tail_col: str, 
    model_params: Dict[str, Any], 
    cols_to_drop: List[str]
) -> pd.DataFrame:
    """
    Trains a single Random Forest model with given parameters and returns
    predictions on the evaluation set.
    """
    # --- 1. Prepare and Check the Test Data ---
    ground_truth_set = set(zip(test_pos_pairs[head_col], test_pos_pairs[tail_col]))
    test_candidate_pairs = set(zip(eval_candidates[head_col], eval_candidates[tail_col]))

    if not ground_truth_set.issubset(test_candidate_pairs):
        missing_pairs = len(ground_truth_set - test_candidate_pairs)
        raise ValueError(
            f"{missing_pairs} pairs from 'test_pos_pairs' are missing "
            "from the 'eval_candidates' candidate DataFrame."
        )

    # --- 2. Prepare Training Data ---
    train_pos = train_pos.assign(label=1)
    train_neg = train_neg.assign(label=0)
    train_df = pd.concat([train_pos, train_neg], ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

    final_cols_to_drop = [head_col, tail_col] + (cols_to_drop or [])
    feature_cols = [col for col in train_df.columns if col not in final_cols_to_drop + ['label']]
    X_train = train_df[feature_cols]
    y_train = train_df['label']

    # --- 3. Train the Random Forest Model ---
    model = RandomForestClassifier(n_estimators=200, random_state=69, **model_params)
    model.fit(X_train, y_train)

    # --- 4. Prepare Test Set and Predict Scores ---
    eval_df = eval_candidates.copy()
    eval_df['true_label'] = eval_df.apply(
        lambda row: (row[head_col], row[tail_col]) in ground_truth_set,
        axis=1
    )
    X_test = eval_df[feature_cols]
    prob_scores = model.predict_proba(X_test)[:, 1]
    eval_df['model_score'] = prob_scores

    return eval_df

In [68]:
def ensemble_evaluation_by_quantile(
    best_configs: pd.DataFrame, 
    train_test_feats: Dict[str, Dict[str, pd.DataFrame]],
    test_pos_pairs: pd.DataFrame,
    grid_search_results: Dict[str, Any],
    head_col: str, 
    tail_col: str, 
    param_col, cols_to_drop: List[str], 
    hits_k_vals: List[int],
    n_bootstrap: int = 1000
) -> Dict[str, Any]:
    """
    Trains and evaluates an ensemble of models, each optimized for a specific
    quantile of a feature, using a different feature matrix for each model config.
    """
    
    # 1. Clean up quantile boundaries
    best_configs = best_configs.sort_values('slice_num').reset_index(drop=True)

    for i in range(1, len(best_configs)):
        prev_max = best_configs.loc[i - 1, 'max_param']
        if best_configs.loc[i, 'min_param'] != prev_max + 1:
            best_configs.loc[i, 'min_param'] = prev_max + 1
    best_configs.loc[len(best_configs) - 1, 'max_param'] = np.inf

    # 2. Iterate through each quantile, train, and predict
    all_predictions = pd.DataFrame()
    results = {}

    test_pos_pairs_with_param = pd.merge(
    test_pos_pairs, train_test_feats['negs_to_test'][best_configs['param_combo'].iloc[0]][[head_col, tail_col, param_col]].drop_duplicates(), 
    on=[head_col, tail_col], 
    how='left'
)
    for idx, row in tqdm(best_configs.iterrows(), total=len(best_configs)):
        slice_num = row['slice_num']
        min_param, max_param = row['min_param'], row['max_param']

        config_key = row['param_combo']
        
        try:
            model_params = grid_search_results[config_key][2]
            # Fetch the FULL training and test data for this config
            train_pos = train_test_feats['train_pos'][config_key]
            train_neg = train_test_feats['train_neg'][config_key]
            eval_candidates = train_test_feats['negs_to_test'][config_key]
        except KeyError:
            print(f"Warning: Data or hyperparameters not found for config '{config_key}'. Skipping.")
            continue

        # Filter test candidates and true positive pairs for the current quantile

        if int(slice_num) == 10:
            test_slice_condition = (eval_candidates[param_col] >= min_param)
            test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param)
        else:
            test_slice_condition = (eval_candidates[param_col] >= min_param) & (eval_candidates[param_col] <= max_param)
            test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param) & (test_pos_pairs_with_param[param_col] <= max_param)
        
        test_slice = eval_candidates[test_slice_condition].copy()
        test_pos_slice = test_pos_pairs_with_param[test_pos_slice_condition].copy()

        # Train and predict
        if not test_slice.empty:
            slice_predictions = train_and_predict_single_model(
                train_pos=train_pos,         # Pass the FULL training data
                train_neg=train_neg,         # Pass the FULL training data
                eval_candidates=test_slice,  # Evaluate on the SLICED test data
                test_pos_pairs=test_pos_slice, # Evaluate on the SLICED true positives
                head_col=head_col, 
                tail_col=tail_col, 
                model_params=model_params, 
                cols_to_drop=cols_to_drop
            )
            all_predictions = pd.concat([all_predictions, slice_predictions], ignore_index=True)
            
            # Evaluate individual slice
            for k in hits_k_vals:
                hits_k, hits_k_ci = calculate_hits_at_k(slice_predictions, k, head_col, tail_col)
                results[f'slice_{slice_num}_Hits@{k}'] = hits_k
                results[f'slice_{slice_num}_Hits@{k}_ci'] = hits_k_ci
            
            # NEW: Calculate recall for the individual slice
            # The function now uses the 'true_label' column already in slice_predictions
            slice_recall, slice_recall_ci = calculate_recall(slice_predictions, head_col, tail_col)
            results[f'slice_{slice_num}_Recall'] = slice_recall
            results[f'slice_{slice_num}_Recall_ci'] = slice_recall_ci
    
    # 3. Evaluate Ensemble
    for k in hits_k_vals:
        hits_k, hits_k_ci = calculate_hits_at_k(all_predictions, k, head_col, tail_col)
        results[f'ensemble_Hits@{k}'] = hits_k
        results[f'ensemble_Hits@{k}_ci'] = hits_k_ci
    
    # NEW: Calculate recall for the entire ensemble
    # The function now uses the 'true_label' column already in all_predictions
    ensemble_recall, ensemble_recall_ci = calculate_recall(all_predictions, head_col, tail_col)
    results['ensemble_Recall'] = ensemble_recall
    results['ensemble_Recall_ci'] = ensemble_recall_ci
        
    return results, all_predictions

### INF ensemble & INF single best

In [ ]:
test_pos_pairs_load = load_pickle(str(config.TEST_POS_PAIRS_PKL))
test_pos_pairs_load

In [54]:
inf_hitk_train_test_feats = load_pickle(str(config.HITK_PARTITIONS_PKL))

In [ ]:
hitk_inf_ensemble, preds_inf_ensemble = ensemble_evaluation_by_quantile(best_configs_l3, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20])

In [ ]:
hitk_inf_ensemble['ensemble_Hits@10']

In [110]:
from pathlib import Path as _Path
if _Path(config.SINGLE_BEST_HITK_PKL).exists():
    single_best_hitk = load_pickle(str(config.SINGLE_BEST_HITK_PKL))
    single_best_recall = load_pickle(str(config.SINGLE_BEST_RECALL_PKL))
    print(f'Pre-computed — loaded both vars from existing pickles')
else:
    single_best_hitk = train_and_predict_single_model(inf_hitk_train_test_feats['train_pos'][3], inf_hitk_train_test_feats['train_neg'][3],
                                                inf_hitk_train_test_feats['negs_to_test'][3], 
                                                test_pos_pairs_load, 
                                                'Gene', 'ADR', 
                                                inf_cv_gs_results_l3[3][2], ['Gene_ADR_deg'])

    single_best_recall = train_and_predict_single_model(inf_hitk_train_test_feats['train_pos'][20], inf_hitk_train_test_feats['train_neg'][20],
                                                inf_hitk_train_test_feats['negs_to_test'][20], 
                                                test_pos_pairs_load, 
                                                'Gene', 'ADR', 
                                                inf_cv_gs_results_l3[20][2], ['Gene_ADR_deg'])

In [116]:
if not (config.SINGLE_BEST_HITK_PKL).exists():
    save_pickle(single_best_hitk, str(config.SINGLE_BEST_HITK_PKL))
    save_pickle(single_best_recall, str(config.SINGLE_BEST_RECALL_PKL))
else:
    print(f'Pre-computed — pickles already exist, skipping saves')

### Benchmarks - keep for CIs

In [117]:
from pathlib import Path as _Path
if _Path(config.RAW_PREDS_PKL).exists():
    raw_preds = load_pickle(str(config.RAW_PREDS_PKL))
    print(f'Pre-computed — loaded {config.RAW_PREDS_PKL.name}')
else:
    raw_preds = train_and_predict_single_model(inf_hitk_train_test_feats['train_pos'][0], inf_hitk_train_test_feats['train_neg'][0],
                                                inf_hitk_train_test_feats['negs_to_test'][0], 
                                                test_pos_pairs_load, 
                                                'Gene', 'ADR', 
                                                inf_cv_gs_results_l3[0][2], ['Gene_ADR_deg'])

In [120]:
if not (config.RAW_PREDS_PKL).exists():
    save_pickle(raw_preds, str(config.RAW_PREDS_PKL))
else:
    print(f'Pre-computed — {config.RAW_PREDS_PKL.name} already exists, skipping save')

In [121]:
from pathlib import Path as _Path
if _Path(config.DWPC_PREDS_PKL).exists():
    dwpc_preds = load_pickle(str(config.DWPC_PREDS_PKL))
    print(f'Pre-computed — loaded {config.DWPC_PREDS_PKL.name}')
else:
    dwpc_preds = train_and_predict_single_model(dwpc_hitk_train_test_feats['train_pos'], dwpc_hitk_train_test_feats['train_neg'],
                                                dwpc_hitk_train_test_feats['negs_to_test'], 
                                                test_pos_pairs_load, 
                                                'Gene', 'ADR', 
                                                dwpc_cv_gs_results_l3_recall[2], ['Gene_ADR_deg'])

In [122]:
if not (config.DWPC_PREDS_PKL).exists():
    save_pickle(dwpc_preds, str(config.DWPC_PREDS_PKL))
else:
    print(f'Pre-computed — {config.DWPC_PREDS_PKL.name} already exists, skipping save')

### Ideal ensemble

In [123]:
def find_best_configs_by_score(
    best_configs_df: pd.DataFrame,
    train_test_feats: Dict[str, Dict[str, pd.DataFrame]],
    test_pos_pairs: pd.DataFrame,
    grid_search_results: Dict[str, Any],
    head_col: str, 
    tail_col: str, 
    param_col, 
    cols_to_drop: List[str], 
    hits_k_vals: List[int],
    optimization_metric: str, # 'hits@k' or 'recall'
    n_bootstrap: int = 1000
) -> Dict[str, Any]:
    """
    Trains every config feature matrix and returns the best scores (overall and per slice) 
    evaluated on a specified metric (Hits@k or Recall).
    """

    # --- 1. Initialize data structures for tracking best scores ---
    
    # Store best scores and CIs for the specified optimization metric
    best_overall_score = {'score': -1.0, 'config': None, 'ci': (np.nan, np.nan)}
    best_overall_predictions = pd.DataFrame()
    
    best_slice_scores = {}
    if optimization_metric == 'hits@k':
        for i in range(1, len(best_configs_df['slice_num'].unique()) + 1):
            for k in hits_k_vals:
                best_slice_scores[f'slice_{i}_Hits@{k}'] = {'score': -1.0, 'config': None, 'ci': (np.nan, np.nan)}
    elif optimization_metric == 'recall':
        for i in range(1, len(best_configs_df['slice_num'].unique()) + 1):
            best_slice_scores[f'slice_{i}_Recall'] = {'score': -1.0, 'config': None, 'ci': (np.nan, np.nan)}
    else:
        raise ValueError("optimization_metric must be 'hits@k' or 'recall'")

    all_configs = list(train_test_feats['train_pos'].keys())

    # 2. Clean up quantile boundaries
    best_configs = best_configs_df.sort_values('slice_num').reset_index(drop=True)
    
    # Pre-process test_pos_pairs once to get the param_col
    test_pos_pairs_with_param = pd.merge(
        test_pos_pairs, 
        train_test_feats['negs_to_test'][all_configs[0]][[head_col, tail_col, param_col]].drop_duplicates(), 
        on=[head_col, tail_col], 
        how='left'
    )
    
    # 3. Main loop over all available configurations
    for config_key in tqdm(all_configs):
        try:
            model_params = grid_search_results[config_key][2]
            train_pos = train_test_feats['train_pos'][config_key]
            train_neg = train_test_feats['train_neg'][config_key]
            eval_candidates = train_test_feats['negs_to_test'][config_key]
        except KeyError:
            print(f"Warning: Data or hyperparameters not found for config '{config_key}'. Skipping.")
            continue

        current_config_predictions = pd.DataFrame()
        
        # Internal loop for each quantile slice
        for idx, row in best_configs.iterrows():
            slice_num = int(row['slice_num'])
            min_param, max_param = row['min_param'], row['max_param']

            if idx > 0 and best_configs.loc[idx, 'min_param'] != best_configs.loc[idx-1, 'max_param'] + 1:
                 min_param = best_configs.loc[idx-1, 'max_param'] + 1
            if idx == len(best_configs_df) - 1:
                max_param = np.inf
            
            if max_param == np.inf:
                test_slice_condition = (eval_candidates[param_col] >= min_param)
                test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param)
            else:
                test_slice_condition = (eval_candidates[param_col] >= min_param) & (eval_candidates[param_col] < max_param)
                test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param) & (test_pos_pairs_with_param[param_col] < max_param)

            test_slice = eval_candidates[test_slice_condition].copy()
            test_pos_slice = test_pos_pairs_with_param[test_pos_slice_condition].copy()

            if not test_slice.empty:
                slice_predictions = train_and_predict_single_model(
                    train_pos=train_pos, 
                    train_neg=train_neg, 
                    eval_candidates=test_slice, 
                    test_pos_pairs=test_pos_slice, 
                    head_col=head_col, 
                    tail_col=tail_col, 
                    model_params=model_params, 
                    cols_to_drop=cols_to_drop
                )
                
                slice_predictions = pd.merge(
                    slice_predictions,
                    test_slice[[head_col, tail_col, param_col]],
                    on=[head_col, tail_col],
                    how='left'
                )

                current_config_predictions = pd.concat([current_config_predictions, slice_predictions], ignore_index=True)
                
                # Update best slice score based on optimization metric
                if optimization_metric == 'hits@k':
                    for k in hits_k_vals:
                        hits_k_score, hits_k_ci = calculate_hits_at_k(
                            slice_predictions, k, head_col, tail_col, n_bootstrap
                        )
                        if hits_k_score > best_slice_scores[f'slice_{slice_num}_Hits@{k}']['score']:
                            best_slice_scores[f'slice_{slice_num}_Hits@{k}']['score'] = hits_k_score
                            best_slice_scores[f'slice_{slice_num}_Hits@{k}']['config'] = config_key
                            best_slice_scores[f'slice_{slice_num}_Hits@{k}']['ci'] = hits_k_ci
                elif optimization_metric == 'recall':
                    recall_score, recall_ci = calculate_recall(
                        slice_predictions, head_col, tail_col, n_bootstrap
                    )
                    if recall_score > best_slice_scores[f'slice_{slice_num}_Recall']['score']:
                        best_slice_scores[f'slice_{slice_num}_Recall']['score'] = recall_score
                        best_slice_scores[f'slice_{slice_num}_Recall']['config'] = config_key
                        best_slice_scores[f'slice_{slice_num}_Recall']['ci'] = recall_ci
                        
        # 4. Evaluate and update best overall scores
        if not current_config_predictions.empty:
            current_score = -1.0
            current_ci = (np.nan, np.nan)
            
            if optimization_metric == 'hits@k':
                # For hits@k, we use the first value in hits_k_vals for the overall best score
                k = hits_k_vals[0]
                current_score, current_ci = calculate_hits_at_k(
                    current_config_predictions, k, head_col, tail_col, n_bootstrap
                )
            elif optimization_metric == 'recall':
                current_score, current_ci = calculate_recall(
                    current_config_predictions, head_col, tail_col, n_bootstrap
                )
            
            if current_score > best_overall_score['score']:
                best_overall_score['score'] = current_score
                best_overall_score['config'] = config_key
                best_overall_score['ci'] = current_ci
                best_overall_predictions = current_config_predictions.copy()
    
    # 5. Format and return final results
    final_results = {}
    if optimization_metric == 'hits@k':
        final_results[f'best_overall_Hits@{hits_k_vals[0]}'] = best_overall_score
        final_results[f'best_overall_predictions_Hits@{hits_k_vals[0]}'] = best_overall_predictions
    elif optimization_metric == 'recall':
        final_results['best_overall_Recall'] = best_overall_score
        final_results['best_overall_predictions_Recall'] = best_overall_predictions

    final_results['best_slice_scores'] = best_slice_scores
    
    return final_results

In [ ]:
from pathlib import Path as _Path
if _Path(config.IDEAL_INF_RES_PKL).exists():
    ideal_ensemble_results = load_pickle(str(config.IDEAL_INF_RES_PKL))
    print(f'Pre-computed — loaded {config.IDEAL_INF_RES_PKL.name}')
else:
    ideal_ensemble_results = find_best_configs_by_score(best_configs_l3, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20])

In [1470]:
if not (config.IDEAL_INF_RES_PKL).exists():
    save_pickle(ideal_ensemble_results, str(config.IDEAL_INF_RES_PKL))
else:
    print(f'Pre-computed — {config.IDEAL_INF_RES_PKL.name} already exists, skipping save')

In [ ]:
ideal_ensemble_results['best_overall_Hits@10']

## Slice-wise comparison

In [129]:
def config_evaluation_by_quantile(
    best_configs: pd.DataFrame, # used for boundaries
    chosen_config: str,
    train_test_feats: Dict[str, Dict[str, pd.DataFrame]],
    test_pos_pairs: pd.DataFrame,
    grid_search_results: Dict[str, Any],
    head_col: str, 
    tail_col: str, 
    param_col: str, 
    cols_to_drop: List[str], 
    hits_k_vals: List[int],
    evaluation_metric: str = 'hits@k', # 'hits@k' or 'recall'
    n_bootstrap: int = 1000
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """
    Trains and evaluates a single model configuration across all quantile slices,
    returning results based on the specified evaluation metric.
    """
    
    # 1. Clean up quantile boundaries
    best_configs = best_configs.sort_values('slice_num').reset_index(drop=True)

    for i in range(1, len(best_configs)):
        prev_max = best_configs.loc[i - 1, 'max_param']
        if best_configs.loc[i, 'min_param'] != prev_max + 1:
            best_configs.loc[i, 'min_param'] = prev_max + 1
    best_configs.loc[len(best_configs) - 1, 'max_param'] = np.inf

    # 2. Iterate through each quantile, train, and predict
    all_predictions = pd.DataFrame()
    results = {}

    test_pos_pairs_with_param = pd.merge(
    test_pos_pairs, train_test_feats['negs_to_test'][best_configs['param_combo'].iloc[0]][[head_col, tail_col, param_col]].drop_duplicates(), 
    on=[head_col, tail_col], 
    how='left'
)
    for idx, row in tqdm(best_configs.iterrows(), total=len(best_configs)):
        slice_num = row['slice_num']
        min_param, max_param = row['min_param'], row['max_param']

        config_key = chosen_config
        
        try:
            model_params = grid_search_results[config_key][2]
            train_pos = train_test_feats['train_pos'][config_key]
            train_neg = train_test_feats['train_neg'][config_key]
            eval_candidates = train_test_feats['negs_to_test'][config_key]
        except KeyError:
            print(f"Warning: Data or hyperparameters not found for config '{config_key}'. Skipping.")
            continue

        if int(slice_num) == 10:
            test_slice_condition = (eval_candidates[param_col] >= min_param)
            test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param)
        else:
            test_slice_condition = (eval_candidates[param_col] >= min_param) & (eval_candidates[param_col] <= max_param)
            test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param) & (test_pos_pairs_with_param[param_col] <= max_param)
        
        test_slice = eval_candidates[test_slice_condition].copy()
        test_pos_slice = test_pos_pairs_with_param[test_pos_slice_condition].copy()

        if not test_slice.empty:
            slice_predictions = train_and_predict_single_model(
                train_pos=train_pos, 
                train_neg=train_neg, 
                eval_candidates=test_slice,  
                test_pos_pairs=test_pos_slice, 
                head_col=head_col, 
                tail_col=tail_col, 
                model_params=model_params, 
                cols_to_drop=cols_to_drop
            )
            all_predictions = pd.concat([all_predictions, slice_predictions], ignore_index=True)
            
            # --- Evaluate individual slice based on evaluation_metric ---
            if evaluation_metric == 'hits@k':
                for k in hits_k_vals:
                    hits_k, hits_k_ci = calculate_hits_at_k(slice_predictions, k, head_col, tail_col, n_bootstrap)
                    results[f'slice_{slice_num}_Hits@{k}'] = hits_k
                    results[f'slice_{slice_num}_Hits@{k}_ci'] = hits_k_ci
            elif evaluation_metric == 'recall':
                recall, recall_ci = calculate_recall(slice_predictions, head_col, tail_col, n_bootstrap)
                results[f'slice_{slice_num}_Recall'] = recall
                results[f'slice_{slice_num}_Recall_ci'] = recall_ci
            else:
                raise ValueError("evaluation_metric must be 'hits@k' or 'recall'")

    # 3. Evaluate Ensemble
    if evaluation_metric == 'hits@k':
        for k in hits_k_vals:
            hits_k, hits_k_ci = calculate_hits_at_k(all_predictions, k, head_col, tail_col, n_bootstrap)
            results[f'ensemble_Hits@{k}'] = hits_k
            results[f'ensemble_Hits@{k}_ci'] = hits_k_ci
    elif evaluation_metric == 'recall':
        recall, recall_ci = calculate_recall(all_predictions, head_col, tail_col, n_bootstrap)
        results[f'ensemble_Recall'] = recall
        results[f'ensemble_Recall_ci'] = recall_ci
        
    return results, all_predictions

In [ ]:
hitk_inf_config_20_hitk, preds_inf_config_20_hitk = config_evaluation_by_quantile(best_configs_l3, 20, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20])

In [ ]:
hitk_inf_config_20_recall, preds_inf_config_20_recall = config_evaluation_by_quantile(best_configs_l3, 20, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20], 'recall')

In [ ]:
from pathlib import Path as _Path
if _Path(config.INF_CONFIGS_SLICE_RECALL_PKL).exists():
    inf_configs_slice_recall = load_pickle(str(config.INF_CONFIGS_SLICE_RECALL_PKL))
    print(f'Pre-computed — loaded {config.INF_CONFIGS_SLICE_RECALL_PKL.name}')
else:
    inf_configs_slice_recall = {combo: None for combo, _ in enumerate(param_combos_load)}

    for combo, _ in enumerate(param_combos_load):

        inf_configs_slice_recall[combo] = config_evaluation_by_quantile(best_configs_l3, combo, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20], 'recall')

In [185]:
if not (config.INF_CONFIGS_SLICE_RECALL_PKL).exists():
    save_pickle(inf_configs_slice_recall, str(config.INF_CONFIGS_SLICE_RECALL_PKL))
else:
    print(f'Pre-computed — {config.INF_CONFIGS_SLICE_RECALL_PKL.name} already exists, skipping save')

In [ ]:
def get_best_slice_configs( # from above output
    inf_configs_slice_results: Dict[str, Any], 
    best_configs_df: pd.DataFrame, 
    metric_name: str
) -> Dict[str, Any]:
    """
    Analyzes evaluation results from a set of model configurations and returns
    the best-performing config for each slice based on the specified metric.

    Args:
        inf_configs_slice_results: A dictionary where keys are config names 
                                   (e.g., 'combo_0') and values are the output 
                                   dictionaries from config_evaluation_by_quantile.
        best_configs_df: The DataFrame defining the slices.
        metric_name: The name of the metric to use for comparison (e.g., 'Recall').

    Returns:
        A dictionary in the format of find_best_configs_by_score, containing
        the best-performing config for each slice.
    """
    
    # Initialize the best_slice_scores dictionary to an empty dict
    best_slice_scores = {}

    # Iterate through all configurations and their results
    for config, results_tuple in inf_configs_slice_results.items():
        results_dict = results_tuple[0]
        # Iterate through the results of each slice for the current config
        for key, value in results_dict.items():
            # Check if the key corresponds to the metric for a slice and is not a CI
            if f'slice_' in key and metric_name in key and '_ci' not in key:
                slice_key = key
                slice_score = value
                
                # Get the corresponding CI
                ci_key = f'{slice_key}_ci'
                ci_value = results_dict.get(ci_key, (float('nan'), float('nan')))

                # Initialize the key in best_slice_scores if it doesn't exist
                if slice_key not in best_slice_scores:
                    best_slice_scores[slice_key] = {'score': -1.0, 'config': None, 'ci': (float('nan'), float('nan'))}

                # Compare with the current best score for that slice
                if slice_score > best_slice_scores[slice_key]['score']:
                    best_slice_scores[slice_key]['score'] = slice_score
                    best_slice_scores[slice_key]['config'] = config
                    best_slice_scores[slice_key]['ci'] = ci_value
    
    # Return the results in the required format
    return {'best_slice_scores': best_slice_scores}

In [206]:
from pathlib import Path as _Path
if _Path(config.IDEAL_INF_RES_RECALL_PKL).exists():
    ideal_ensemble_results_recall = load_pickle(str(config.IDEAL_INF_RES_RECALL_PKL))
    print(f'Pre-computed — loaded {config.IDEAL_INF_RES_RECALL_PKL.name}')
else:
    ideal_ensemble_results_recall = get_best_slice_configs(inf_configs_slice_recall, best_configs_l3, 'Recall')

In [207]:
if not (config.IDEAL_INF_RES_RECALL_PKL).exists():
    save_pickle(ideal_ensemble_results_recall, str(config.IDEAL_INF_RES_RECALL_PKL))
else:
    print(f'Pre-computed — {config.IDEAL_INF_RES_RECALL_PKL.name} already exists, skipping save')

#### CV Ensemble

#### Raw

In [ ]:
hitk_inf_config_0_hitk, preds_inf_config_0_hitk = config_evaluation_by_quantile(best_configs_l3, 0, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20])
hitk_inf_config_0_recall, preds_inf_config_recall = config_evaluation_by_quantile(best_configs_l3, 0, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20], 'recall')

In [ ]:
print(hitk_inf_config_0_hitk['slice_1.0_Hits@10'])
print(hitk_inf_config_0_hitk['slice_2.0_Hits@10'])

#### DWPC

In [139]:
inf_hitk_train_test_feats['train_pos']['dwpc'] = dwpc_hitk_train_test_feats['train_pos']
inf_hitk_train_test_feats['train_neg']['dwpc'] = dwpc_hitk_train_test_feats['train_neg']
inf_hitk_train_test_feats['negs_to_test']['dwpc'] = dwpc_hitk_train_test_feats['negs_to_test']

In [140]:
dwpc_cv_gs_hyperparams = {'dwpc': {}}
dwpc_cv_gs_hyperparams['dwpc'][2] = dwpc_cv_gs_results_l3_recall[2]

In [ ]:
hitk_inf_config_dwpc_hitk, preds_inf_config_dwpc_hitk = config_evaluation_by_quantile(best_configs_l3, 'dwpc', inf_hitk_train_test_feats, test_pos_pairs_load, dwpc_cv_gs_hyperparams, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20])
hitk_inf_config_dwpc_recall, preds_inf_config_dwpc_recall = config_evaluation_by_quantile(best_configs_l3, 'dwpc', inf_hitk_train_test_feats, test_pos_pairs_load, dwpc_cv_gs_hyperparams, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20], 'recall')

### Ideal ensemble

## Top features - bottom 2 deciles only so bounds correct

In [143]:
from typing import Callable

In [366]:
def _hits_at_k_simple(
    df: pd.DataFrame,
    head_col: str,
    tail_col: str,
    k: int = 10
) -> float:
    """Calculates the filtered Hits@k score."""
    all_filtered_ranks = []
    
    if df.empty:
        return 0.0

    for _, group in df.groupby(head_col):
        sorted_group = group.sort_values('model_score', ascending=False)
        true_tails = set(sorted_group[sorted_group['true_label'] == True][tail_col])

        if not true_tails:
            continue

        ranked_tails = sorted_group[tail_col].tolist()
        for true_t in true_tails:
            try:
                raw_rank_idx = ranked_tails.index(true_t)
            except ValueError:
                continue # True tail not in predictions, treat as lowest rank
            
            num_other_true_tails_ranked_higher = len([
                t for t in ranked_tails[:raw_rank_idx] if t in true_tails
            ])
            
            filtered_rank = (raw_rank_idx + 1) - num_other_true_tails_ranked_higher
            all_filtered_ranks.append(filtered_rank)
    
    if not all_filtered_ranks:
        return 0.0
        
    all_filtered_ranks = np.array(all_filtered_ranks)
    return np.mean(all_filtered_ranks <= k)

In [146]:
feats_l3 = load_pickle(str(config.DRKG_FEATS_L3_PKL))

In [ ]:
inf_hitk_train_test_feats.keys()

In [147]:
all_negs_to_test = {combo_idx: neg_to_test for combo_idx, neg_to_test in inf_hitk_train_test_feats['negs_to_test'].items()}
all_negs_to_test['dwpc'] = dwpc_hitk_train_test_feats['negs_to_test']

In [ ]:
bounds_data = {
    'min_param': [0, 6],
    'max_param': [5, 17]
}
ae_deg_boundaries = pd.DataFrame(bounds_data)

ae_deg_boundaries

In [365]:
from typing import Callable, Dict, List
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

In [ ]:
def permimp_ranking(model, X, y_true, feat_names, scorer):
    """
    Calculates permutation importance and returns the top 10 ranked features.
    
    Args:
        model: A trained model.
        X: DataFrame of features.
        y_true: Series of true labels.
        feat_names: List of feature names.
        scorer: The metric to use for scoring.
    
    Returns:
        A dictionary of the top 10 features and their importance scores.
    """
    
    permimp = permutation_importance(model, X, \
                       y_true, scoring=scorer, 
                       n_repeats=5, random_state=69, n_jobs=-1)['importances_mean']
        
    permimp_dict = {}
    for i in range(len(permimp)):
        permimp_dict[i] = permimp[i]
    
    permimp_rankings = {}
    
    for feat_index in sorted(permimp_dict, 
                             key=lambda dict_key: permimp_dict[dict_key], 
                             reverse=True)[:10]:
        permimp_rankings[feat_names[feat_index]] = round(permimp_dict[feat_index], 3)
        
    return permimp_rankings

In [ ]:
def get_sliced_permutation_importance( # old recall gold
    model: Callable,
    eval_df: pd.DataFrame,
    neg_to_test: pd.DataFrame,
    boundaries_df: pd.DataFrame,
    feature_cols: List[str],
    param_col: str,
    head_col: str,
    tail_col: str,
    k: int = 10,
    optimization_metric: str = 'hits@k',
    top_n_features: int = 5
) -> Dict[int, pd.DataFrame]:
    """
    Calculates permutation importance for features on different data slices, ensuring
    all data conforms to slice boundaries and the param_col is not dropped.
    """
    
    # Check for param_col existence to prevent KeyError
    if param_col not in eval_df.columns:
        raise KeyError(f"'{param_col}' not found in 'eval_df'. It is required for slicing.")
    
    if param_col not in neg_to_test.columns:
        # Assuming the neg_to_test is intended to be joined with eval_df on head/tail,
        # but param_col is used for slicing. It must be in the final merged dataframe.
        # This check is to be explicit.
        raise KeyError(f"'{param_col}' not found in 'neg_to_test'. It is required for merging/slicing.")

    # Apply the boundary adjustment logic as requested, ensuring contiguous ranges
    best_configs = boundaries_df.copy()
    for i in range(1, len(best_configs)):
        prev_max = best_configs.loc[i - 1, 'max_param']
        best_configs.loc[i, 'min_param'] = prev_max + 1

    # Merge dataframes once to get a complete view

    full_eval_data = pd.merge(eval_df, neg_to_test, on=[head_col, tail_col, param_col], how='inner')
    
    if full_eval_data.empty:
        print("Warning: Merged evaluation data is empty. Check input dataframes and column names.")
        return {}

    results = {}
    
    # Define the scorer based on the chosen metric
    def hits_at_k_scorer(y_true, y_pred, **kwargs):
        df = pd.DataFrame({
            'true_label': y_true, 
            'model_score': y_pred, 
            head_col: kwargs.get(head_col), 
            tail_col: kwargs.get(tail_col)
        })
        return calculate_hits_at_k(df, k=k, head_col=head_col, tail_col=tail_col)[0]
        
    def recall_scorer(y_true, y_pred, **kwargs):
        predicted_labels = (y_pred > 0.5).astype(int)
        return recall_score(y_true, predicted_labels)
        
    if optimization_metric == 'hits@k':
        scorer = make_scorer(hits_at_k_scorer, needs_proba=True)
    elif optimization_metric == 'recall':
        scorer = make_scorer(recall_scorer, needs_proba=True)
    else:
        raise ValueError("optimization_metric must be 'hits@k' or 'recall'")

    # Iterate through each slice, slice the full dataframe, and run permutation importance
    for slice_idx, row in best_configs.iterrows():
        min_b, max_b = row['min_param'], row['max_param']

        # Slice the entire merged dataframe to ensure all data conforms to the boundaries
        slice_mask = (full_eval_data[param_col] >= min_b) & (full_eval_data[param_col] <= max_b)
        data_slice = full_eval_data.loc[slice_mask].copy()

        if data_slice.empty:
            continue

        # Isolate features and labels for the current slice
        X_slice = data_slice[feature_cols]
        y_slice = data_slice['true_label']

        # Use the provided helper function to calculate importance
        importance_ranks = permimp_ranking(
            model=model,
            X=X_slice,
            y_true=y_slice,
            feat_names=feature_cols,
            scorer=scorer
        )
        
        # Format and store the results
        importance_df = pd.DataFrame(
            importance_ranks.items(), columns=['feature', 'importance']
        ).sort_values('importance', ascending=False).head(top_n_features).reset_index(drop=True)

        results[slice_idx] = importance_df
        
    return results

In [ ]:
perm_rankings_by_config = {cfg_key: None for cfg_key in hitk_dfs}

for cfg_key in tqdm(hitk_dfs):

    perm_rankings_by_config[cfg_key] = get_sliced_permutation_importance(hitk_dfs[cfg_key][0], 
                                                                        hitk_dfs[cfg_key][1], 
                                                                        all_negs_to_test[cfg_key], 
                                                                        ae_deg_boundaries, 
                                                                        feats_l3, 
                                                                        'Gene_ADR_deg', 'Gene', 'ADR')

In [387]:
results_dir = str(config.FEAT_IMPORTANCE_DIR)
#save_pickle(perm_rankings_by_config, f'{results_dir}/config_top_feats_by_slice.pkl')

In [409]:
def get_sliced_permutation_importance_hitk(
    model: Callable,
    eval_df: pd.DataFrame,
    neg_to_test: pd.DataFrame,
    boundaries_df: pd.DataFrame,
    feature_cols: List[str],
    param_col: str,
    head_col: str,
    tail_col: str,
    k: int = 10,
    optimization_metric: str = 'hits@k',
    top_n_features: int = 5
) -> Dict[int, pd.DataFrame]:
    """
    Calculates permutation importance for features on different data slices, ensuring
    all data conforms to slice boundaries and the param_col is not dropped.
    """
    
    # Check for param_col existence to prevent KeyError
    if param_col not in eval_df.columns:
        raise KeyError(f"'{param_col}' not found in 'eval_df'. It is required for slicing.")
    
    if param_col not in neg_to_test.columns:
        raise KeyError(f"'{param_col}' not found in 'neg_to_test'. It is required for merging/slicing.")

    # Apply the boundary adjustment logic as requested, ensuring contiguous ranges
    best_configs = boundaries_df.copy()
    for i in range(1, len(best_configs)):
        prev_max = best_configs.loc[i - 1, 'max_param']
        best_configs.loc[i, 'min_param'] = prev_max + 1

    # Merge dataframes once to get a complete view
    full_eval_data = pd.merge(eval_df, neg_to_test, on=[head_col, tail_col, param_col], how='inner')
    
    if full_eval_data.empty:
        print("Warning: Merged evaluation data is empty. Check input dataframes and column names.")
        return {}

    results = {}
    
    # Define the scorer based on the chosen metric
    # The 'recall' scorer is correct as it is point-wise, so we keep the original logic.
    if optimization_metric == 'recall':
        scorer = 'recall'
    elif optimization_metric == 'hits@k':
        # For 'hits@k', we cannot use make_scorer. The logic must be handled manually
        # to ensure the necessary data (head, tail, etc.) is available for ranking.
        pass # The scorer is not needed here; logic is implemented below
    else:
        raise ValueError("optimization_metric must be 'hits@k' or 'recall'")

    # Iterate through each slice, slice the full dataframe, and run permutation importance
    for slice_idx, row in best_configs.iterrows():
        min_b, max_b = row['min_param'], row['max_param']

        # Slice the entire merged dataframe to ensure all data conforms to the boundaries
        slice_mask = (full_eval_data[param_col] >= min_b) & (full_eval_data[param_col] <= max_b)
        data_slice = full_eval_data.loc[slice_mask].copy()

        if data_slice.empty:
            continue

        # Isolate features and labels for the current slice
        X_slice = data_slice[feature_cols]
        y_slice = data_slice['true_label']

        # Use the correct logic for each metric
        if optimization_metric == 'recall':
            # This logic works for recall as it's a simple, point-wise metric
            importance_ranks = permimp_ranking(
                model=model,
                X=X_slice,
                y_true=y_slice,
                feat_names=feature_cols,
                scorer=scorer
            )
        elif optimization_metric == 'hits@k':
            # This manual logic correctly passes the necessary context for the hits@k scorer
            # as seen in the working code.
            base_scores = model.predict_proba(X_slice)[:, 1]
            scored_slice = data_slice.assign(model_score=base_scores)
            
            baseline_score = calculate_hits_at_k(scored_slice, k, head_col, tail_col)[0]
            
            importances = []
            for feature in feature_cols:
                X_shuffled = X_slice.copy()
                X_shuffled[feature] = np.random.permutation(X_shuffled[feature].values)
                
                shuffled_probs = model.predict_proba(X_shuffled)[:, 1]
                shuffled_scored_slice = data_slice.assign(model_score=shuffled_probs)
                
                score = calculate_hits_at_k(shuffled_scored_slice, k, head_col, tail_col)[0]
                
                importance = baseline_score - score
                importances.append({'feature': feature, 'importance': importance})
                
            importance_ranks = {item['feature']: item['importance'] for item in importances}

        # Format and store the results
        importance_df = pd.DataFrame(
            importance_ranks.items(), columns=['feature', 'importance']
        ).sort_values('importance', ascending=False).head(top_n_features).reset_index(drop=True)

        results[slice_idx] = importance_df
        
    return results

In [ ]:
perm_rankings_by_config = {cfg_key: {'recall': None, 'hits@10': None} for cfg_key in hitk_dfs} # only consider recall from these

for cfg_key in tqdm(hitk_dfs):

    perm_rankings_by_config[cfg_key]['recall'] = get_sliced_permutation_importance(hitk_dfs[cfg_key][0], 
                                                                        hitk_dfs[cfg_key][1], 
                                                                        all_negs_to_test[cfg_key], 
                                                                        ae_deg_boundaries, 
                                                                        feats_l3,
                                                                        'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                        optimization_metric='recall')
    
    perm_rankings_by_config[cfg_key]['hits@10'] = get_sliced_permutation_importance(hitk_dfs[cfg_key][0], 
                                                                    hitk_dfs[cfg_key][1], 
                                                                    all_negs_to_test[cfg_key], 
                                                                    ae_deg_boundaries, 
                                                                    feats_l3,
                                                                    'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                    optimization_metric='hits@k')
    save_pickle(perm_rankings_by_config, f'{results_dir}/config_top_feats_by_slice.pkl')

In [ ]:
for cfg_key in tqdm(hitk_dfs):
    
    perm_rankings_by_config[cfg_key]['hits@10'] = get_sliced_permutation_importance_hitk(hitk_dfs[cfg_key][0], 
                                                                    hitk_dfs[cfg_key][1], 
                                                                    all_negs_to_test[cfg_key], 
                                                                    ae_deg_boundaries, 
                                                                    feats_l3,
                                                                    'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                    optimization_metric='hits@k')
    save_pickle(perm_rankings_by_config, f'{results_dir}/config_top_feats_by_slice_hitk.pkl')

#### Top slice bounds

In [ ]:
bounds_data2 = {
    'min_param': [133, 150],
    'max_param': [149, 256]
}
ae_deg_boundaries2 = pd.DataFrame(bounds_data2)

ae_deg_boundaries2

In [ ]:
perm_rankings_topdec_tpr_only = {cfg_key: None for cfg_key in hitk_dfs} # top dec for comparison

for cfg_key in tqdm(hitk_dfs):

    perm_rankings_topdec_tpr_only[cfg_key] = get_sliced_permutation_importance(hitk_dfs[cfg_key][0], 
                                                                        hitk_dfs[cfg_key][1], 
                                                                        all_negs_to_test[cfg_key], 
                                                                        ae_deg_boundaries2, 
                                                                        feats_l3,
                                                                        'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                        optimization_metric='recall')
    

    save_pickle(perm_rankings_topdec_tpr_only, f'{results_dir}/config_top_feats_by_slice_topdec_recall.pkl')

### Single feature training of Raw and INF_12

#### Raw

#### 12

In [ ]:
# --- feature importance setup variables ---
feature1 = '(n_source:Gene)-[r1:Gene_ADR]->(n_1:ADR)<-[r2:Gene_ADR]-(n_2:Gene)-[r3:Gene_ADR]->(n_target:ADR)'
feature2 = '(n_source:Gene)-[r1:Gene_Compound]->(n_1:Compound)<-[r2:Gene_Compound]-(n_2:Gene)-[r3:Gene_ADR]->(n_target:ADR)'
feature3 = '(n_source:Gene)-[r1:Gene_MolecularFunction]->(n_1:MolecularFunction)<-[r2:Gene_MolecularFunction]-(n_2:Gene)-[r3:Gene_ADR]->(n_target:ADR)'
feature4 = '(n_source:Gene)-[r1:Gene_BiologicalProcess]->(n_1:BiologicalProcess)<-[r2:Gene_BiologicalProcess]-(n_2:Gene)-[r3:Gene_ADR]->(n_target:ADR)'
thresh = 149
thresh2 = 6
_12_train_feats = pd.concat([train_test_pkl['pos_train'][12], train_test_pkl['neg_train'][12]])
_12_y_train = pd.concat([
    pd.Series([1]*len(train_test_pkl['pos_train'][12])),
    pd.Series([0]*len(train_test_pkl['neg_train'][12]))
]).reset_index(drop=True)
_raw_y_train = _12_y_train  # alias used in some cells


In [ ]:
'ADR'

_12_train_one_feat = np.array(_12_train_feats[feature1]).reshape(-1, 1)

_12_model_1feat = RandomForestClassifier(n_estimators=200, random_state=69) # **inf_cv_gs_results_l3[0][2]

_12_model_1feat.fit(_12_train_one_feat, _raw_y_train)

# bottom decile
_12_test_feats_0 =  all_negs_to_test[12].loc[all_negs_to_test[12]['Gene_ADR_deg'] < thresh2][feats_l3]

_12_test_feats_0 = np.array(_12_test_feats_0[feature1]).reshape(-1, 1)

_12_test_label_0 =  hitk_dfs[12][1].loc[hitk_dfs[12][1]['Gene_ADR_deg'] < thresh2]['true_label']

_12_10_y_pred_1feat_0 = _12_model_1feat.predict(_12_test_feats_0)

recall_score(_12_test_label_0, _12_10_y_pred_1feat_0)

In [ ]:
'Compound'

_12_train_one_feat = np.array(_12_train_feats[feature2]).reshape(-1, 1)

_12_model_1feat = RandomForestClassifier(n_estimators=200, random_state=69) # **inf_cv_gs_results_l3[0][2]

_12_model_1feat.fit(_12_train_one_feat, _raw_y_train)

# top decile
_12_test_feats_0 =  all_negs_to_test[12].loc[all_negs_to_test[12]['Gene_ADR_deg'] < thresh2][feats_l3]

_12_test_feats_0 = np.array(_12_test_feats_0[feature2]).reshape(-1, 1)

_12_test_label_0 =  hitk_dfs[12][1].loc[hitk_dfs[12][1]['Gene_ADR_deg'] < thresh2]['true_label']

_12_10_y_pred_1feat_0 = _12_model_1feat.predict(_12_test_feats_0)

recall_score(_12_test_label_0, _12_10_y_pred_1feat_0)

In [ ]:
'Mol Func'

_12_train_one_feat = np.array(_12_train_feats[feature3]).reshape(-1, 1)

_12_model_1feat = RandomForestClassifier(n_estimators=200, random_state=69) # **inf_cv_gs_results_l3[0][2]

_12_model_1feat.fit(_12_train_one_feat, _raw_y_train)

# top decile
_12_test_feats_0 =  all_negs_to_test[12].loc[all_negs_to_test[12]['Gene_ADR_deg'] < thresh2][feats_l3]

_12_test_feats_0 = np.array(_12_test_feats_0[feature3]).reshape(-1, 1)

_12_test_label_0 =  hitk_dfs[12][1].loc[hitk_dfs[12][1]['Gene_ADR_deg'] < thresh2]['true_label']

_12_10_y_pred_1feat_0 = _12_model_1feat.predict(_12_test_feats_0)

recall_score(_12_test_label_0, _12_10_y_pred_1feat_0)

In [ ]:
'Bio process'

_12_train_one_feat = np.array(_12_train_feats[feature4]).reshape(-1, 1)

_12_model_1feat = RandomForestClassifier(n_estimators=200, random_state=69) # **inf_cv_gs_results_l3[0][2]

_12_model_1feat.fit(_12_train_one_feat, _raw_y_train)

# top decile
_12_test_feats_0 =  all_negs_to_test[12].loc[all_negs_to_test[12]['Gene_ADR_deg'] < thresh2][feats_l3]

_12_test_feats_0 = np.array(_12_test_feats_0[feature4]).reshape(-1, 1)

_12_test_label_0 =  hitk_dfs[12][1].loc[hitk_dfs[12][1]['Gene_ADR_deg'] < thresh2]['true_label']

_12_10_y_pred_1feat_0 = _12_model_1feat.predict(_12_test_feats_0)

recall_score(_12_test_label_0, _12_10_y_pred_1feat_0)

### Top feature reviews - slice 1 - Use for thesis
#### Hit @ 10

### TPR

### Top feature reviews - slice 10
#### Hit @ 10

#### Recall

#### Top feature reviews - slice 2

## Charts to plot from Internal testing

In [346]:
def plot_hitk_ranked_bar_chart(results_df: pd.DataFrame, title: str, x_max_factor):
    """
    Generates a single horizontal bar chart showing a ranked list of models
    from a DataFrame of results, including bootstrap confidence intervals.

    Args:
        results_df (pd.DataFrame): A DataFrame with 'Model', 'Hits@10', 
                                   'Hits@10 CI lower', and 'Hits@10 CI upper'
                                   columns.
        title (str): The title for the chart.
    """
    # Create an 'error' column representing the margin of error
    # This is the difference between the mean and the lower CI bound
    results_df = results_df.copy()
    results_df['error'] = results_df['Hits@10'] - results_df['Hits@10 CI lower']
    
    # Sort the DataFrame by the main metric
    sorted_df = results_df.sort_values('Hits@10', ascending=True)
    
    # 3. Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.get_cmap('winter', len(sorted_df))

    # Create the horizontal bar plot with error bars
    # The 'xerr' argument now uses the calculated error margin
    bars = ax.barh(sorted_df['model'], sorted_df['Hits@10'], 
                   xerr=sorted_df['error'], capsize=5,
                   color=colors(np.linspace(0.25, 0.9, len(sorted_df))))
    
    # 4. Format the chart
    ax.set_title(title, fontsize=18)
    ax.set_xlabel('Hits@10', fontsize=14)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=12)
    
    # Adjust x-axis limits to prevent cutoff and show the full error bar
    # Use the max of both the value and the upper CI for a correct limit
    x_max = sorted_df['Hits@10 CI upper'].max()
    ax.set_xlim(left=0, right=x_max * x_max_factor)

    plt.tight_layout()
    plt.show()

In [ ]:
hitk_plot_data_internal_test = {'model': ['INF_CV_Best\n(Ensemble)', 'Original', 'DWPC'],
                                'Hits@10': [hitk_inf_ensemble['ensemble_Hits@10'],
                                            calculate_hits_at_k(raw_preds, 10, 'Gene', 'ADR')[0],
                                            calculate_hits_at_k(dwpc_preds, 10, 'Gene', 'ADR')[0]],
                                'Hits@10 CI lower': [hitk_inf_ensemble['ensemble_Hits@10_ci'][0],
                                                     calculate_hits_at_k(raw_preds, 10, 'Gene', 'ADR')[1][0],
                                                     calculate_hits_at_k(dwpc_preds, 10, 'Gene', 'ADR')[1][0]],
                                'Hits@10 CI upper': [hitk_inf_ensemble['ensemble_Hits@10_ci'][1],
                                                     calculate_hits_at_k(raw_preds, 10, 'Gene', 'ADR')[1][1],
                                                     calculate_hits_at_k(dwpc_preds, 10, 'Gene', 'ADR')[1][1]]
                                }

pd.DataFrame(hitk_plot_data_internal_test)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_internal_test), 'Hits @ 10 on Entire Holdout Test Set', 1.45)

In [348]:
def plot_recall_ranked_bar_chart(results_df: pd.DataFrame, title: str, x_max_factor): # recall
    """
    Generates a single horizontal bar chart showing a ranked list of models
    from a DataFrame of results, including bootstrap confidence intervals.
    """
    # Create an 'error' column representing the margin of error
    # This is the difference between the mean and the lower CI bound
    results_df = results_df.copy()
    results_df['error'] = results_df['Recall'] - results_df['Recall CI lower']
    
    # Sort the DataFrame by the main metric
    sorted_df = results_df.sort_values('Recall', ascending=True)
    
    # 3. Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.get_cmap('winter', len(sorted_df))

    # Create the horizontal bar plot with error bars
    # The 'xerr' argument now uses the calculated error margin
    bars = ax.barh(sorted_df['model'], sorted_df['Recall'], 
                   xerr=sorted_df['error'], capsize=5,
                   color=colors(np.linspace(0.25, 0.9, len(sorted_df))))
    
    # 4. Format the chart
    ax.set_title(title, fontsize=18)
    ax.set_xlabel('TPR', fontsize=14)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=12)
    
    # Adjust x-axis limits to prevent cutoff and show the full error bar
    # Use the max of both the value and the upper CI for a correct limit
    x_max = sorted_df['Recall CI upper'].max()
    ax.set_xlim(left=0, right=x_max * x_max_factor)

    plt.tight_layout()
    plt.show()

In [ ]:
recall_plot_data_internal_test = {'model': ['INF_CV_Best\n(Ensemble)', 'Original', 'DWPC'],
                                'Recall': [hitk_inf_ensemble['ensemble_Recall'],
                                            calculate_recall(raw_preds, 'Gene', 'ADR')[0],
                                            calculate_recall(dwpc_preds, 'Gene', 'ADR')[0]],
                                'Recall CI lower': [hitk_inf_ensemble['ensemble_Recall_ci'][0],
                                                     calculate_recall(raw_preds, 'Gene', 'ADR')[1][0],
                                                     calculate_recall(dwpc_preds, 'Gene', 'ADR')[1][0]],
                                'Recall CI upper': [hitk_inf_ensemble['ensemble_Recall_ci'][1],
                                                     calculate_recall(raw_preds, 'Gene', 'ADR')[1][1],
                                                     calculate_recall(dwpc_preds, 'Gene', 'ADR')[1][1]]
                                }

pd.DataFrame(recall_plot_data_internal_test)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_internal_test), 'TPR on Entire Holdout Test Set', 1.15)

In [ ]:
print(hitk_inf_ensemble['slice_1.0_Hits@10'],
      hitk_inf_ensemble['slice_1.0_Hits@10_ci'])

print(hitk_inf_config_0['slice_1.0_Hits@10'],
      hitk_inf_config_0['slice_1.0_Hits@10_ci'])

print(hitk_inf_config_dwpc['slice_1.0_Hits@10'],
      hitk_inf_config_dwpc['slice_1.0_Hits@10_ci'])
print(ideal_ensemble_results['best_slice_scores']['slice_1_Hits@10'])

In [ ]:
hitk_plot_data_internal_test_d1 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Hits@10': [hitk_inf_ensemble['slice_1.0_Hits@10'],
                                            hitk_inf_config_0_hitk['slice_1.0_Hits@10'],
                                            hitk_inf_config_dwpc_hitk['slice_1.0_Hits@10']],
                                'Hits@10 CI lower': [hitk_inf_ensemble['slice_1.0_Hits@10_ci'][0],
                                                     hitk_inf_config_0_hitk['slice_1.0_Hits@10_ci'][0],
                                                     hitk_inf_config_dwpc_hitk['slice_1.0_Hits@10_ci'][0]],
                                'Hits@10 CI upper': [hitk_inf_ensemble['slice_1.0_Hits@10_ci'][1],
                                                     hitk_inf_config_0_hitk['slice_1.0_Hits@10_ci'][1],
                                                     hitk_inf_config_dwpc_hitk['slice_1.0_Hits@10_ci'][1]]
                                }

pd.DataFrame(hitk_plot_data_internal_test_d1)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_internal_test_d1), 'Hits @ 10 on Holdout Test Data with 0-5 AEs per Target\n(1st Decile Defined in Cross-Validation)', 1.2)

In [ ]:
recall_plot_data_internal_test_d1 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Recall': [hitk_inf_ensemble['slice_1.0_Recall'],
                                            hitk_inf_config_0_recall['slice_1.0_Recall'],
                                            hitk_inf_config_dwpc_recall['slice_1.0_Recall']],
                                'Recall CI lower': [hitk_inf_ensemble['slice_1.0_Recall_ci'][0],
                                                     hitk_inf_config_0_recall['slice_1.0_Recall_ci'][0],
                                                     hitk_inf_config_dwpc_recall['slice_1.0_Recall_ci'][0]],
                                'Recall CI upper': [hitk_inf_ensemble['slice_1.0_Recall_ci'][1],
                                                     hitk_inf_config_0_recall['slice_1.0_Recall_ci'][1],
                                                     hitk_inf_config_dwpc_recall['slice_1.0_Recall_ci'][1]]
                                }

pd.DataFrame(recall_plot_data_internal_test_d1)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_internal_test_d1), 'TPR on Holdout Test Data with 0-5 AEs per Target\n(1st Decile Defined in Cross-Validation)', 1.2)

In [449]:
d2_title_hitk  = 'Hits @ 10 on Holdout Test Data with 6-17 AEs per Target\n(2nd Decile Defined in Cross-Validation)'
d2_title_recall  = 'TPR on Holdout Test Data with 6-17 AEs per Target\n(2nd Decile Defined in Cross-Validation)'

In [ ]:
hitk_plot_data_internal_test_d2 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Hits@10': [hitk_inf_ensemble['slice_2.0_Hits@10'],
                                            hitk_inf_config_0_hitk['slice_2.0_Hits@10'],
                                            hitk_inf_config_dwpc_hitk['slice_2.0_Hits@10']],
                                'Hits@10 CI lower': [hitk_inf_ensemble['slice_2.0_Hits@10_ci'][0],
                                                     hitk_inf_config_0_hitk['slice_2.0_Hits@10_ci'][0],
                                                     hitk_inf_config_dwpc_hitk['slice_2.0_Hits@10_ci'][0]],
                                'Hits@10 CI upper': [hitk_inf_ensemble['slice_2.0_Hits@10_ci'][1],
                                                     hitk_inf_config_0_hitk['slice_2.0_Hits@10_ci'][1],
                                                     hitk_inf_config_dwpc_hitk['slice_2.0_Hits@10_ci'][1]]
                                }

pd.DataFrame(hitk_plot_data_internal_test_d2)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_internal_test_d2), d2_title_hitk, 1.2)

In [ ]:
recall_plot_data_internal_test_d2 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Recall': [hitk_inf_ensemble['slice_2.0_Recall'],
                                            hitk_inf_config_0_recall['slice_2.0_Recall'],
                                            hitk_inf_config_dwpc_recall['slice_2.0_Recall']],
                                'Recall CI lower': [hitk_inf_ensemble['slice_2.0_Recall_ci'][0],
                                                     hitk_inf_config_0_recall['slice_2.0_Recall_ci'][0],
                                                     hitk_inf_config_dwpc_recall['slice_2.0_Recall_ci'][0]],
                                'Recall CI upper': [hitk_inf_ensemble['slice_2.0_Recall_ci'][1],
                                                     hitk_inf_config_0_recall['slice_2.0_Recall_ci'][1],
                                                     hitk_inf_config_dwpc_recall['slice_2.0_Recall_ci'][1]]
                                }

pd.DataFrame(recall_plot_data_internal_test_d2)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_internal_test_d2), d2_title_recall, 1.2)

# III. Soldatos

## Preparation

In [ ]:
soldatos2_df = pd.read_csv(str(config.SOLDATOS_FILE_2), sep='	')

soldatos2_df.head()

In [ ]:
soldatos3_df = pd.read_csv(str(config.SOLDATOS_FILE_3), sep='	')

soldatos3_df.head()

In [ ]:
soldatos4_df = pd.read_csv(str(config.SOLDATOS_FILE_4), sep='	')

soldatos4_df.head()

In [ ]:
soldatos_df = pd.concat([
    soldatos2_df.loc[soldatos2_df['Q-Value (Based on Fisher Exact Test)'] <= 0.05],
    soldatos3_df.loc[soldatos3_df['Q-Value (Based on Fisher Exact Test)'] <= 0.05],
    soldatos4_df.loc[soldatos4_df['Q-Value (Based on Fisher Exact Test)'] <= 0.05]

])

soldatos_df.shape

In [260]:
soldatos_gold = soldatos_df.loc[(soldatos_df['NUM_CASES_TARGET_AND_REACTION'] >= 10) & (soldatos_df['PRR'] >= 2)]

In [1039]:
map_ = pd.read_csv(str(config.MAPPING_FILE), sep='\t')

In [ ]:
hitk_dfs_mapped = {cfg_key: None for cfg_key in hitk_dfs}

for cfg_key in tqdm(hitk_dfs, total=len(hitk_dfs)):

    hitk_dfs_mapped[cfg_key] = hitk_dfs[cfg_key][1].merge(map_, left_on='Gene', right_on='id').drop(['id', 'kind'], axis=1).rename(columns={'name':'Gene_name'})

    hitk_dfs_mapped[cfg_key] = hitk_dfs_mapped[cfg_key].merge(map_, left_on='ADR', right_on='id', how='left').drop_duplicates().drop(['id','kind'], axis=1).rename(columns={'name':'ADR_name'})

In [ ]:
soldatos_gold['ADR_name'] = soldatos_gold['REACTION_NAME']

soldatos_gold['ADR_name']

In [269]:
def filter_and_process_series(df, col_name, newcol_name, suffix):
    """
    Filters a DataFrame to keep rows where a specific column's values
    end with a given suffix, and creates a new series with the suffix removed.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to check for the suffix.
        suffix (str): The suffix to check for and remove.

    Returns:
        tuple: A tuple containing:
            - A new DataFrame with only the rows that had the suffix.
            - A new Series with the suffix removed from the entries.
    """
    # Create a boolean mask to identify rows where the series entry ends with the suffix
    has_suffix_mask = df[col_name].str.endswith(suffix)
    
    # Filter the DataFrame using the mask to keep only the relevant rows
    filtered_df = df[has_suffix_mask].copy()
    
    # Create a new series from the filtered DataFrame's column,
    # with the suffix removed from each entry
    filtered_df[newcol_name] = filtered_df[col_name].str.removesuffix(suffix)
    
    return filtered_df

In [282]:
soldatos_gold_genes = filter_and_process_series(soldatos_gold, 'SWISSPROT_ID', 'Gene_name', '_HUMAN')

In [ ]:
soldatos_gold_genes.info()

In [284]:
def capitalise_column(df, column_name):
    """
    Capitalizes the first letter of each entry in a specified column of a DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to be capitalized.

    Returns:
        pd.DataFrame: A new DataFrame with the specified column's entries capitalized.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    capitalized_df = df.copy()
    
    # Use the .str.capitalize() method to capitalize the first letter of each string
    # in the specified column
    capitalized_df[column_name] = capitalized_df[column_name].str.capitalize()
    
    return capitalized_df

In [ ]:
for cfg_key, df in tqdm(hitk_dfs_mapped.items(), total=len(hitk_dfs_mapped)):

    hitk_dfs_mapped[cfg_key] = capitalise_column(df, 'ADR_name')

In [ ]:
from pathlib import Path as _Path
if _Path(config.SOLDATOS_INTERSECTS_PKL).exists():
    intersects = load_pickle(str(config.SOLDATOS_INTERSECTS_PKL))
    new_test = load_pickle(str(config.SOLDATOS_NEW_TEST_PKL))
    print(f'Pre-computed — loaded both vars from existing pickles')
else:
    intersects = {cfg_key: None for cfg_key in hitk_dfs_mapped}

    new_test = {cfg_key: None for cfg_key in hitk_dfs_mapped}

    for cfg_key in tqdm(hitk_dfs_mapped, total=len(hitk_dfs_mapped)):

        intersects[cfg_key] = pd.merge(hitk_dfs_mapped[cfg_key], soldatos_gold_genes, on=['Gene_name', 'ADR_name'], how='inner')

        new_test[cfg_key] = intersects[cfg_key].loc[intersects[cfg_key]['true_label']==False]

In [287]:
if not (config.SOLDATOS_INTERSECTS_PKL).exists():
    save_pickle(intersects, str(config.SOLDATOS_INTERSECTS_PKL))
    save_pickle(new_test, str(config.SOLDATOS_NEW_TEST_PKL))
else:
    print(f'Pre-computed — pickles already exist, skipping saves')

In [ ]:
from pathlib import Path as _Path
if _Path(config.SOLDATOS_HITK_MAPPED_PKL).exists():
    hitk_dfs_mapped_newtest = load_pickle(str(config.SOLDATOS_HITK_MAPPED_PKL))
    print(f'Pre-computed — loaded {config.SOLDATOS_HITK_MAPPED_PKL.name}')
else:
    hitk_dfs_mapped_newtest = {cfg_key: None for cfg_key in hitk_dfs_mapped}

    for cfg_key in tqdm(hitk_dfs_mapped_newtest, total=len(hitk_dfs_mapped_newtest)):

        hitk_dfs_mapped_newtest[cfg_key] = hitk_dfs_mapped[cfg_key][hitk_dfs_mapped[0]['true_label'] != True].copy()

        matched_keys = pd.merge(hitk_dfs_mapped_newtest[cfg_key], new_test[0], on=['Gene_name','ADR_name'], how='inner')

        matching_mask = pd.MultiIndex.from_frame(hitk_dfs_mapped_newtest[cfg_key][['Gene_name', 'ADR_name']]).isin(
            pd.MultiIndex.from_frame(matched_keys[['Gene_name', 'ADR_name']])
        )
    
        hitk_dfs_mapped_newtest[cfg_key].loc[matching_mask, 'true_label'] = True

In [1115]:
if not (config.SOLDATOS_HITK_MAPPED_PKL).exists():
    save_pickle(hitk_dfs_mapped_newtest, str(config.SOLDATOS_HITK_MAPPED_PKL))
else:
    print(f'Pre-computed — {config.SOLDATOS_HITK_MAPPED_PKL.name} already exists, skipping save')

## Evaluating on Soldatos

### Numbers assocs in 1st & 2nd deciles

### Prev eval - good

In [246]:
hitk_dfs_mapped_newtest = load_pickle(str(config.SOLDATOS_HITK_MAPPED_PKL))

In [248]:
map_ = pd.read_csv(str(config.MAPPING_FILE), sep='\t')

In [ ]:
preds_inf_ensemble_mapped = preds_inf_ensemble[['Gene','ADR','Gene_ADR_deg','true_label','model_score']] # find the ensemble defined in internal testing

preds_inf_ensemble_mapped = preds_inf_ensemble_mapped.merge(map_, left_on='Gene', right_on='id').drop(['id', 'kind'], axis=1).rename(columns={'name':'Gene_name'})

preds_inf_ensemble_mapped = preds_inf_ensemble_mapped.merge(map_, left_on='ADR', right_on='id', how='left').drop_duplicates().drop(['id','kind'], axis=1).rename(columns={'name':'ADR_name'})

preds_inf_ensemble_mapped = capitalise_column(preds_inf_ensemble_mapped, 'ADR_name')
preds_inf_ensemble_mapped

In [295]:
hitk_dfs_mapped_newtest['inf_ensemble'] = preds_inf_ensemble_mapped[preds_inf_ensemble_mapped['true_label'] != True].copy()

matched_keys = pd.merge(hitk_dfs_mapped_newtest['inf_ensemble'], new_test[0][['Gene_name','ADR_name']], on=['Gene_name','ADR_name'], how='inner')

matching_mask = pd.MultiIndex.from_frame(hitk_dfs_mapped_newtest['inf_ensemble'][['Gene_name', 'ADR_name']]).isin(
    pd.MultiIndex.from_frame(matched_keys[['Gene_name', 'ADR_name']])
)

hitk_dfs_mapped_newtest['inf_ensemble'].loc[matching_mask, 'true_label'] = True

In [ ]:
hitk_dfs_mapped_newtest['inf_ensemble'].loc[hitk_dfs_mapped_newtest['inf_ensemble']['true_label']==True].shape

In [297]:
if not (config.SOLDATOS_PREDS_MAPPED_PKL).exists():
    save_pickle(hitk_dfs_mapped_newtest, str(config.SOLDATOS_PREDS_MAPPED_PKL))
else:
    print(f'Pre-computed — {config.SOLDATOS_PREDS_MAPPED_PKL.name} already exists, skipping save')

In [ ]:
from pathlib import Path as _Path
if _Path(config.SOLDATOS_RESULTS_PKL).exists():
    newtest_overall_at10_ci = load_pickle(str(config.SOLDATOS_RESULTS_PKL))
    print(f'Pre-computed — loaded {config.SOLDATOS_RESULTS_PKL.name}')
else:
    newtest_overall_at10_ci = {cfg_key: {'Hits@10': None, 'Recall': None} for cfg_key in hitk_dfs_mapped_newtest} # default k is 10

    for cfg_key in tqdm(hitk_dfs_mapped_newtest, total=len(hitk_dfs_mapped_newtest)):

        newtest_overall_at10_ci[cfg_key]['Hits@10'] = calculate_hits_at_k(hitk_dfs_mapped_newtest[cfg_key], 10, 'Gene', 'ADR')

        newtest_overall_at10_ci[cfg_key]['Recall'] = calculate_recall(hitk_dfs_mapped_newtest[cfg_key], 'Gene', 'ADR')

In [300]:
if not (config.SOLDATOS_RESULTS_PKL).exists():
    save_pickle(newtest_overall_at10_ci, str(config.SOLDATOS_RESULTS_PKL))
else:
    print(f'Pre-computed — {config.SOLDATOS_RESULTS_PKL.name} already exists, skipping save')

In [311]:
def compute_sliced_hits_at_k_and_recall( # assumes no open-ended last boundary ie excludes top decile
    head_col: str, 
    tail_col: str,
    boundaries_df: pd.DataFrame,
    eval_df: pd.DataFrame,
    param_col_name: str,
    k: int = 10
) -> pd.DataFrame:
    """
    Computes Hits@k and Recall with CIs for slices of data defined by parameter
    boundaries, using a contiguous boundary adjustment.

    Args:
        head_col (str): Name of the head column.
        tail_col (str): Name of the tail column.
        boundaries_df (pd.DataFrame): DataFrame with 'min_param' and
                                      'max_param' columns defining the slices.
        eval_df (pd.DataFrame): The evaluation data, containing the parameter
                                column to slice on and model scores.
        param_col_name (str): The name of the column in eval_df to slice by.
        k (int, optional): The 'k' value for Hits@k. Defaults to 10.

    Returns:
        pd.DataFrame: A DataFrame with Hits@k, Recall, and their CIs for each slice.
    """
    if boundaries_df.empty:
        return pd.DataFrame()
    
    for i in range(1, len(boundaries_df)):
        prev_max = boundaries_df.loc[i - 1, 'max_param']
        if boundaries_df.loc[i, 'min_param'] != prev_max + 1:
            boundaries_df.loc[i, 'min_param'] = prev_max + 1

    results_list = []
    
    # Iterate through slices and compute metrics
    for idx, row in boundaries_df.iterrows():
        min_param = row['min_param']
        max_param = row['max_param']
        
        # Slicing with adjusted boundaries
        data_slice = eval_df[(eval_df[param_col_name] >= min_param) & (eval_df[param_col_name] <= max_param)]

        slice_result = {
            'slice_idx': idx,
            'min_param': min_param,
            'max_param': max_param
        }

        if not data_slice.empty:
            # Compute Hits@k and its CI
            hits_k, hits_k_ci = calculate_hits_at_k(data_slice, k, head_col, tail_col)
            slice_result[f'Hits@{k}'] = hits_k
            slice_result[f'Hits@{k}_ci_lower'] = hits_k_ci[0]
            slice_result[f'Hits@{k}_ci_upper'] = hits_k_ci[1]

            # Compute Recall and its CI
            recall, recall_ci = calculate_recall(data_slice, head_col, tail_col)
            slice_result['Recall'] = recall
            slice_result['Recall_ci_lower'] = recall_ci[0]
            slice_result['Recall_ci_upper'] = recall_ci[1]
        else:
            slice_result[f'Hits@{k}'] = np.nan
            slice_result[f'Hits@{k}_ci_lower'] = np.nan
            slice_result[f'Hits@{k}_ci_upper'] = np.nan
            slice_result['Recall'] = np.nan
            slice_result['Recall_ci_lower'] = np.nan
            slice_result['Recall_ci_upper'] = np.nan

        results_list.append(slice_result)

    return pd.DataFrame(results_list)

In [ ]:
bounds_data = {
    'min_param': [0, 6],
    'max_param': [5, 17]
}
ae_deg_boundaries = pd.DataFrame(bounds_data)

ae_deg_boundaries

In [ ]:
per_slice_at10_newtest = {cfg_key: None for cfg_key in hitk_dfs_mapped_newtest} # for k = 10

for cfg_key in tqdm(hitk_dfs_mapped_newtest, total=len(hitk_dfs_mapped_newtest)):
    
    per_slice_at10_newtest[cfg_key] = compute_sliced_hits_at_k_and_recall('Gene', 'ADR',
                                                           ae_deg_boundaries,
                                                           hitk_dfs_mapped_newtest[cfg_key],
                                                           'Gene_ADR_deg')

## Charts to plot from External testing

### Results to plot

In [ ]:
hitk_plot_data_external_test = {'model': ['INF_CV_Best\n(Ensemble)', 'Original', 'DWPC'],
                                'Hits@10': [newtest_overall_at10_ci['inf_ensemble']['Hits@10'][0],
                                            newtest_overall_at10_ci[0]['Hits@10'][0],
                                            newtest_overall_at10_ci['dwpc']['Hits@10'][0]],
                                'Hits@10 CI lower': [newtest_overall_at10_ci['inf_ensemble']['Hits@10'][1][0],
                                            newtest_overall_at10_ci[0]['Hits@10'][1][0],
                                            newtest_overall_at10_ci['dwpc']['Hits@10'][1][0]],
                                'Hits@10 CI upper': [newtest_overall_at10_ci['inf_ensemble']['Hits@10'][1][1],
                                            newtest_overall_at10_ci[0]['Hits@10'][1][1],
                                            newtest_overall_at10_ci['dwpc']['Hits@10'][1][1]]
                                }

pd.DataFrame(hitk_plot_data_external_test)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_external_test), 'Hits @ 10 on Entire External Test Set', 3)

In [ ]:
recall_plot_data_external_test = {'model': ['INF_CV_Best\n(Ensemble)', 'Original', 'DWPC'],
                                'Recall': [newtest_overall_at10_ci['inf_ensemble']['Recall'][0],
                                            newtest_overall_at10_ci[0]['Recall'][0],
                                            newtest_overall_at10_ci['dwpc']['Recall'][0]],
                                'Recall CI lower': [newtest_overall_at10_ci['inf_ensemble']['Recall'][1][0],
                                            newtest_overall_at10_ci[0]['Recall'][1][0],
                                            newtest_overall_at10_ci['dwpc']['Recall'][1][0]],
                                'Recall CI upper': [newtest_overall_at10_ci['inf_ensemble']['Recall'][1][1],
                                            newtest_overall_at10_ci[0]['Recall'][1][1],
                                            newtest_overall_at10_ci['dwpc']['Recall'][1][1]]
                                }

pd.DataFrame(recall_plot_data_external_test)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_external_test), 'TPR on Entire External Test Set', 1.75)

### Top features Soldatos

In [ ]:
perm_rankings_soldatos = {cfg_key: {'hits@10': None, 'recall': None} for cfg_key in [12, 0, 'dwpc']}

for cfg_key in tqdm([12, 0, 'dwpc'], total=3):
    
    print('doing hits@10...')
    perm_rankings_soldatos[cfg_key]['hits@10'] = get_sliced_permutation_importance_hitk(hitk_dfs[cfg_key][0], 
                                                                    hitk_dfs_mapped_newtest[cfg_key], 
                                                                    all_negs_to_test[cfg_key], 
                                                                    ae_deg_boundaries, 
                                                                    feats_l3,
                                                                    'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                    optimization_metric='hits@k')
    
    print('doing recall...')
    perm_rankings_soldatos[cfg_key]['recall'] = get_sliced_permutation_importance_hitk(hitk_dfs[cfg_key][0], 
                                                                hitk_dfs_mapped_newtest[cfg_key], 
                                                                all_negs_to_test[cfg_key], 
                                                                ae_deg_boundaries, 
                                                                feats_l3,
                                                                'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                optimization_metric='recall')
    
    #save_pickle(perm_rankings_by_config, f'{results_dir}/config_top_feats_by_slice_soldatos.pkl')

In [ ]:
hitk_plot_data_external_test_d1 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Hits@10': [per_slice_at10_newtest[12].iloc[[0]]['Hits@10'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Hits@10'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Hits@10'].values[0]],


                                'Hits@10 CI lower': [per_slice_at10_newtest[12].iloc[[0]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Hits@10_ci_lower'].values[0]],

                                            
                                'Hits@10 CI upper': [per_slice_at10_newtest[12].iloc[[0]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Hits@10_ci_upper'].values[0]],

                                }

pd.DataFrame(hitk_plot_data_external_test_d1)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_external_test_d1), 'Hits @ 10 on External Test Data with 0-5 AEs per Target\n(1st Decile Defined in Cross-Validation)', 3)

In [ ]:
recall_plot_data_external_test_d1 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Recall': [per_slice_at10_newtest[12].iloc[[0]]['Recall'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Recall'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Recall'].values[0]],


                                'Recall CI lower': [per_slice_at10_newtest[12].iloc[[0]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Recall_ci_lower'].values[0]],

                                            
                                'Recall CI upper': [per_slice_at10_newtest[12].iloc[[0]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Recall_ci_upper'].values[0]],

                                }

pd.DataFrame(recall_plot_data_external_test_d1)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_external_test_d1), 'TPR on External Test Data with 0-5 AEs per Target\n(1st Decile Defined in Cross-Validation)', 2.5)

In [ ]:
hitk_plot_data_external_test_d2 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Hits@10': [per_slice_at10_newtest[7].iloc[[1]]['Hits@10'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Hits@10'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Hits@10'].values[0]],

                                'Hits@10 CI lower': [per_slice_at10_newtest[7].iloc[[1]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Hits@10_ci_lower'].values[0]],
                                            
                                'Hits@10 CI upper': [per_slice_at10_newtest[7].iloc[[1]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Hits@10_ci_upper'].values[0]]
                                }

pd.DataFrame(hitk_plot_data_external_test_d2)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_external_test_d2), 'Hits @ 10 on External Test Data with 6-17 AEs per Target\n(2nd Decile Defined in Cross-Validation)', 3)

In [ ]:
recall_plot_data_external_test_d2 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Recall': [per_slice_at10_newtest[7].iloc[[1]]['Recall'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Recall'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Recall'].values[0]],

                                'Recall CI lower': [per_slice_at10_newtest[7].iloc[[1]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Recall_ci_lower'].values[0]],
                                            
                                'Recall CI upper': [per_slice_at10_newtest[7].iloc[[1]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Recall_ci_upper'].values[0]]
                                }

pd.DataFrame(recall_plot_data_external_test_d2)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_external_test_d2), 'TPR on External Test Data with 6-17 AEs per Target\n(2nd Decile Defined in Cross-Validation)', 1.5)

## Figure S4 — Synthesis: Holdout vs. External Test Performance
Bar chart comparing Hits@10 and TPR for all three methods on the holdout and external (Soldatos) test sets.
Illustrates the collapse of hits@10 on external data contrasted with INF's resilient TPR.

In [ ]:
# ---------------------------------------------------------------------------
# Figure S4: Holdout vs. External Test Performance (synthesis plot)
# Compares Hits@10 and TPR for INF Ensemble, DWPC, and Original features
# across the holdout test set (n=998) and Soldatos external test set (n=7,804).
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Compute point estimates and CIs ---
raw_hitk_res  = calculate_hits_at_k(raw_preds,  10, 'Gene', 'ADR')
dwpc_hitk_res = calculate_hits_at_k(dwpc_preds, 10, 'Gene', 'ADR')
raw_rec_res   = calculate_recall(raw_preds,  'Gene', 'ADR')
dwpc_rec_res  = calculate_recall(dwpc_preds, 'Gene', 'ADR')

data = {
    'Hits@10': {
        'holdout': [
            (hitk_inf_ensemble['ensemble_Hits@10'],        hitk_inf_ensemble['ensemble_Hits@10_ci']),
            (dwpc_hitk_res[0],                               dwpc_hitk_res[1]),
            (raw_hitk_res[0],                                raw_hitk_res[1]),
        ],
        'external': [
            (newtest_overall_at10_ci['inf_ensemble']['Hits@10'][0], newtest_overall_at10_ci['inf_ensemble']['Hits@10'][1]),
            (newtest_overall_at10_ci['dwpc']['Hits@10'][0],         newtest_overall_at10_ci['dwpc']['Hits@10'][1]),
            (newtest_overall_at10_ci[0]['Hits@10'][0],               newtest_overall_at10_ci[0]['Hits@10'][1]),
        ],
    },
    'TPR': {
        'holdout': [
            (hitk_inf_ensemble['ensemble_Recall'],          hitk_inf_ensemble['ensemble_Recall_ci']),
            (dwpc_rec_res[0],                                 dwpc_rec_res[1]),
            (raw_rec_res[0],                                  raw_rec_res[1]),
        ],
        'external': [
            (newtest_overall_at10_ci['inf_ensemble']['Recall'][0], newtest_overall_at10_ci['inf_ensemble']['Recall'][1]),
            (newtest_overall_at10_ci['dwpc']['Recall'][0],         newtest_overall_at10_ci['dwpc']['Recall'][1]),
            (newtest_overall_at10_ci[0]['Recall'][0],               newtest_overall_at10_ci[0]['Recall'][1]),
        ],
    },
}

methods = ['INF Ensemble', 'DWPC', 'Original']
colors  = ['#1976D2', '#F57C00', '#757575']   # blue, orange, grey
x       = np.arange(len(methods))
width   = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=False)

for ax, metric in zip(axes, ['Hits@10', 'TPR']):
    hold_vals = [v for v, _ in data[metric]['holdout']]
    hold_errs = ([v - ci[0] for v, ci in data[metric]['holdout']],
                 [ci[1] - v for v, ci in data[metric]['holdout']])
    ext_vals  = [v for v, _ in data[metric]['external']]
    ext_errs  = ([v - ci[0] for v, ci in data[metric]['external']],
                 [ci[1] - v for v, ci in data[metric]['external']])

    bars1 = ax.bar(x - width / 2, hold_vals, width,
                   color=colors, alpha=0.90,
                   yerr=hold_errs, capsize=5,
                   ecolor='black', error_kw={'linewidth': 1.2, 'zorder': 5})
    bars2 = ax.bar(x + width / 2, ext_vals, width,
                   color=colors, alpha=0.45, hatch='//',
                   yerr=ext_errs, capsize=5,
                   ecolor='black', error_kw={'linewidth': 1.2, 'zorder': 5})

    # Annotate bar values
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8.5)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8.5)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=11)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

# Shared legend (holdout = solid, external = hatched)
legend_elements = [
    mpatches.Patch(facecolor='grey', alpha=0.90,
                   label='Holdout test set (n = 998)'),
    mpatches.Patch(facecolor='grey', alpha=0.45, hatch='//',
                   label='External test set — Soldatos et al. (n = 7,804)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.06))

fig.suptitle(
    'Holdout vs. External Test Performance\n'
    'Hits@10 collapses on external data; INF TPR remains resilient',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(str(config.FIGS_DIR / 'figS4_holdout_vs_external.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure S4 saved to {config.FIGS_DIR / 'figS4_holdout_vs_external.png'}")